# Surface heat in Brno, 2000–2026, from Landsat
**Question (Brno City Council):** how have rising temperatures affected the city over the last 25 years, and where should adaptation start?

**Approach.** Landsat Collection 2 Level-2 surface temperature (Landsat 5, 7, 8, 9), free and continuous since the 1980s, loaded as a data cube from the Microsoft Planetary Computer STAC catalogue. For every clear summer scene (June–August) the notebook computes the **surface urban heat island (SUHI)** as urban minus rural surface temperature, following Dobrovolný (2013), *Theor. Appl. Climatol.* 112: 89–98. The difference removes much of the weather of each day, although it remains sensitive to the state of the rural reference (for example dry or harvested farmland in August) and to the dates of the available scenes, which section 5b tests explicitly.

**Outputs** (folder `outputs/`, zipped at the end): scene availability; the SUHI time series with its trend and a robustness check against alternative rural references and July-only scenes; maps of the heat anomaly for the first and last period, their difference and the per-pixel trend; a replication of Dobrovolný (2013) for every 5-year period (temperature by land-cover class, class-based SUHI, NDVI and building-density regressions on a 300 m grid); heat by Local Climate Zone; a ranking of the 29 city districts; a comparison with published values from previous studies; and an interactive dashboard comparing an early, a middle and a recent summer, written to `docs/index.html` for GitHub Pages.

**Run:** in Colab use `Runtime → Run all`; locally, install `requirements.txt` and run all cells in Jupyter or VS Code. No account or key is needed. Expect roughly 20–40 minutes, mostly data loading. OpenStreetMap layers come from public Overpass servers; if they are busy the notebook retries on mirror servers and, as a last resort, falls back to ESA WorldCover (check the printed land-cover source). On Windows with PostGIS installed, keep the first cell, which stops rasterio from reading the PostGIS projection database.

In [ ]:
%pip install -q pystac-client planetary-computer odc-stac osmnx rioxarray

In [ ]:
import os
# Some installers (e.g. PostGIS on Windows) point PROJ/GDAL to an old data folder; remove that
# so rasterio and pyproj use their own bundled data. Must run before rasterio is imported.
for var in ("PROJ_LIB", "PROJ_DATA", "GDAL_DATA"):
    os.environ.pop(var, None)

In [ ]:
import os, time, warnings, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats
from rasterio.features import rasterize
import planetary_computer, pystac_client, odc.stac
import osmnx as ox

warnings.filterwarnings("ignore", category=RuntimeWarning)

# OpenStreetMap downloads go through public Overpass servers, which are often busy.
# osm_features() retries with a longer timeout and falls back to mirror servers;
# osmnx caches every answer in ./cache, so a re-run does not download again.
OVERPASS_URLS = ["https://overpass-api.de/api", "https://overpass.kumi.systems/api",
                 "https://overpass.private.coffee/api"]
ox.settings.use_cache = True
ox.settings.requests_timeout = 180

def osm_features(fetch, *args, tries_per_server=1):
    last = None
    for url in OVERPASS_URLS:
        for attempt in range(tries_per_server):
            try:
                print(f"  OpenStreetMap: asking {url} ...", flush=True)
                if hasattr(ox.settings, "overpass_url"):         # osmnx >= 2.0
                    ox.settings.overpass_url = url
                else:                                            # osmnx 1.x
                    ox.settings.overpass_endpoint = url
                return fetch(*args)
            except Exception as e:
                last = e
                print(f"  Overpass {url} attempt {attempt + 1} failed: {type(e).__name__}")
                time.sleep(10 * (attempt + 1))
    raise RuntimeError(f"all Overpass servers failed: {last}")

# ---------------- configuration ----------------
YEARS = list(range(2000, 2027))     # summers 2000–2026
SUMMER = ("06-01", "08-31")
MAX_SCENE_CLOUD = 60           # % cloud for the whole scene (pre-filter)
MIN_CITY_CLEAR = 0.70          # share of the city that must be clear for a scene to be used
L7_LAST_YEAR = 2016            # Landsat 7 orbit drifted to earlier overpasses afterwards
CRS, RES = "EPSG:32633", 30    # UTM 33N, 30 m
RING_KM = (2, 12)              # rural reference: ring outside the city boundary
ELEV_TOL = 40                  # rural pixels within +-40 m of the urban median elevation
PERIODS = {"2000–2004": (2000, 2004), "2005–2009": (2005, 2009), "2010–2014": (2010, 2014),
           "2015–2019": (2015, 2019), "2020–2026": (2020, 2026)}
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
# network: fail instead of hanging forever when a server stops answering, retry a few times
os.environ.update({"GDAL_HTTP_TIMEOUT": "90", "GDAL_HTTP_CONNECTTIMEOUT": "20", "GDAL_HTTP_MAX_RETRY": "5",
                   "GDAL_HTTP_RETRY_DELAY": "3", "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
                   "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF,.tiff"})

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 220, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

## 1. City and district boundaries (OpenStreetMap)
If the district download fails, the notebook continues without the district ranking. An official alternative is the RÚIAN dataset from ČÚZK: upload a GeoJSON of the *městské části* and set `DISTRICTS_FILE`.

In [ ]:
DISTRICTS_FILE = None   # e.g. "brno_districts.geojson" after uploading it to Colab

city = ox.geocode_to_gdf("Brno, Czechia").to_crs(CRS)[["geometry"]]
city_geom = city.geometry.iloc[0]

def load_districts():
    if DISTRICTS_FILE:
        return gpd.read_file(DISTRICTS_FILE).to_crs(CRS)
    for level in ("10", "9"):
        try:
            g = osm_features(ox.features_from_place, "Brno, Czechia", {"boundary": "administrative", "admin_level": level})
        except Exception:
            continue
        g = g[g.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(CRS)
        g = g[g.representative_point().within(city_geom.buffer(50))]
        if "name" in g.columns:
            official = g[g["name"].fillna("").str.startswith("Brno-")]   # the 29 městské části
            if len(official) >= 20:
                g = official
        if len(g) >= 15:
            return g.reset_index()[["name", "geometry"]].drop_duplicates("name")
    return None

districts = load_districts()
print("City area: %.0f km²" % (city_geom.area / 1e6),
      "| districts:", 0 if districts is None else len(districts))

study = city_geom.buffer(RING_KM[1] * 1000)
STUDY_BBOX = tuple(gpd.GeoSeries([study], crs=CRS).to_crs(4326).total_bounds)
CITY_BBOX = tuple(city.to_crs(4326).total_bounds)

## 2. Search the catalogue and check data availability
A scene is *usable* when at least 70 % of the city is free of cloud, cirrus, shadow and fill (e.g. Landsat 7 stripes), using the `qa_pixel` bits. This is checked on a coarse 300 m grid to keep it fast.

In [ ]:
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                    modifier=planetary_computer.sign_inplace)
items = []
for y in YEARS:
    plats = ["landsat-5", "landsat-8", "landsat-9"] + (["landsat-7"] if y <= L7_LAST_YEAR else [])
    res = catalog.search(collections=["landsat-c2-l2"], bbox=STUDY_BBOX,
                         datetime=f"{y}-{SUMMER[0]}/{y}-{SUMMER[1]}",
                         query={"eo:cloud_cover": {"lt": MAX_SCENE_CLOUD}, "platform": {"in": plats}})
    items += [it for it in res.items() if it.properties.get("landsat:collection_category") == "T1"]
print("Candidate scenes:", len(items))

BAD_BITS = (1 << 0) | (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4)   # fill, dilated cloud, cirrus, cloud, shadow
def clear(qa):
    return (qa.astype("uint16") & BAD_BITS) == 0

qa = odc.stac.load(items, bands=["qa_pixel"], bbox=CITY_BBOX, crs=CRS, resolution=300,
                   groupby="solar_day", chunks={})["qa_pixel"]
share = clear(qa).mean(("y", "x")).compute().to_series()
share.index = pd.DatetimeIndex(share.index).normalize()

day = lambda it: pd.Timestamp(it.datetime).tz_localize(None).normalize()
usable_days = set(share[share >= MIN_CITY_CLEAR].index)
usable = [it for it in items if day(it) in usable_days]

inv = pd.DataFrame({"date": [day(it) for it in usable], "platform": [it.properties["platform"] for it in usable]})
inv = inv.drop_duplicates("date").set_index("date").join(share.rename("city_clear"))
inv.to_csv(f"{OUT}/scene_inventory.csv")
print("Usable scenes (dates):", len(inv))

per_year = inv.groupby([inv.index.year, "platform"]).size().unstack(fill_value=0).reindex(YEARS, fill_value=0)
colors = {"landsat-5": "#7a9cc6", "landsat-7": "#c9a227", "landsat-8": "#2f6f5e", "landsat-9": "#9bc4b5"}
ax = per_year.plot(kind="bar", stacked=True, figsize=(10, 3.2), width=0.8,
                   color=[colors[c] for c in per_year.columns])
ax.set(xlabel="", ylabel="Usable scenes", title="Clear summer Landsat scenes over Brno (≥70 % of the city clear)")
ax.legend(frameon=False, ncol=4)
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_scenes_per_year.png"); plt.show()

## 3. Load the data cube (30 m grid): surface temperature and NDVI
**Grid versus true resolution.** The cube uses the 30 m grid of the Collection 2 product, but the thermal sensors see coarser detail: about **120 m** for Landsat 5 TM, **60 m** for Landsat 7 ETM+ and **100 m** for Landsat 8/9 TIRS; USGS resamples them to 30 m. Temperature differences between features smaller than roughly 100 m (a street, a narrow river, a single roof) are therefore blurred, while NDVI (red and near-infrared bands) is truly 30 m. Results are interpreted at the scale of blocks, 300 m grid cells, urban zones and districts, never single pixels.

Landsat 5/7 and 8/9 name their thermal band differently (`lwir` / `lwir11`), so they are loaded separately and stacked on the time axis. Scale factors are those of Collection 2 Level-2. The cube is read **year by year into memory**: Planetary Computer download links expire after about 45 minutes, so each batch is signed again just before it is read, and later sections never go back to the internet.

In [ ]:
import pystac
os.environ.update({"GDAL_HTTP_MAX_RETRY": "5", "GDAL_HTTP_RETRY_DELAY": "2"})

def resign(its):
    # Planetary Computer links expire after ~45 min: strip the old token and sign again
    out = []
    for it in its:
        it = it.clone()
        for a in it.assets.values():
            a.href = a.href.split("?")[0]
        out.append(planetary_computer.sign(it))
    return out

def load_cube(its, thermal):
    # load one batch of scenes into memory: LST in °C and NDVI, cloud-masked
    ds = odc.stac.load(resign(its), bands=[thermal, "red", "nir08", "qa_pixel"], bbox=STUDY_BBOX,
                       crs=CRS, resolution=RES, groupby="solar_day", chunks={})
    ok = clear(ds["qa_pixel"]) & (ds[thermal] > 0)
    lst = (ds[thermal] * 0.00341802 + 149.0 - 273.15).where(ok)
    red = ds["red"] * 0.0000275 - 0.2
    nir = ds["nir08"] * 0.0000275 - 0.2
    ndvi = ((nir - red) / (nir + red)).where(ok)
    return xr.Dataset({"lst": lst, "ndvi": ndvi}).astype("float32").load()

# load year by year, so every batch is read with fresh links
parts = []
for y in YEARS:
    year_items = [it for it in usable if day(it).year == y]
    for plats, thermal in ((("landsat-5", "landsat-7"), "lwir"), (("landsat-8", "landsat-9"), "lwir11")):
        batch = [it for it in year_items if it.properties["platform"] in plats]
        if batch:
            parts.append(load_cube(batch, thermal))
    print(y, "loaded", len(year_items), "items")

cube = xr.concat(parts, dim="time").sortby("time")
cube = cube.sel(time=~pd.DatetimeIndex(cube.time.values).duplicated())
gbox = cube.odc.geobox
print(cube)
print(f"Cube in memory: {cube.nbytes / 1e9:.1f} GB")

## 4. Reference masks: urban, rural, water
* **Urban:** built-up pixels (ESA WorldCover) inside the city boundary.
* **Rural:** cropland or grassland in a 2–12 km ring outside the city, at a similar elevation to the urban area (Copernicus DEM). Brno lies in a basin with forested hills to the west and north; comparing the city with cooler, higher forests would inflate the heat island (Dobrovolný 2013).
* **Water** is excluded everywhere.

In [ ]:
def aux_layer(collection, asset, resampling):
    its = list(catalog.search(collections=[collection], bbox=STUDY_BBOX).items())
    return odc.stac.load(its, bands=[asset], geobox=gbox, resampling=resampling)[asset].max("time")

worldcover = aux_layer("esa-worldcover", "map", "nearest")
dem = aux_layer("cop-dem-glo-30", "data", "bilinear")

def rasterize_geom(geom):
    return xr.DataArray(rasterize([(geom, 1)], out_shape=gbox.shape, transform=gbox.affine, fill=0).astype(bool),
                        coords={"y": cube.y, "x": cube.x}, dims=("y", "x"))

in_city = rasterize_geom(city_geom)
in_study = rasterize_geom(study)
in_ring = rasterize_geom(city_geom.buffer(RING_KM[1] * 1000).difference(city_geom.buffer(RING_KM[0] * 1000)))
water = worldcover == 80
urban = in_city & (worldcover == 50)
z0 = float(dem.where(urban).median())
rural = in_ring & worldcover.isin([30, 40]) & (abs(dem - z0) <= ELEV_TOL)
print(f"Urban median elevation {z0:.0f} m | urban px {int(urban.sum())} | rural px {int(rural.sum())}")

fig, ax = plt.subplots(figsize=(5.5, 5))
(urban.astype(int) + 2 * rural.astype(int)).where(~water).plot(ax=ax, cmap="Set2", add_colorbar=False)
city.boundary.plot(ax=ax, color="k", lw=0.8)
ax.set(title="Urban (green) and rural reference (orange)", xlabel="", ylabel=""); ax.set_aspect("equal")
plt.tight_layout(); plt.savefig(f"{OUT}/fig0_reference_masks.png"); plt.show()

## 5. Heat island intensity for every scene, and its trend
SUHI = median urban surface temperature − median rural surface temperature, per scene. Scenes where either area is less than half observed are dropped. The trend uses the Theil–Sen slope with a Mann–Kendall significance test, both robust to outliers.

In [ ]:
lst = cube["lst"].where(~water)
u_med = lst.where(urban).median(("y", "x"))
r_med = lst.where(rural).median(("y", "x"))
u_cov = lst.where(urban).notnull().sum(("y", "x")) / int(urban.sum())
r_cov = lst.where(rural).notnull().sum(("y", "x")) / int(rural.sum())

series = xr.Dataset({"urban": u_med, "rural": r_med, "u_cov": u_cov, "r_cov": r_cov}).compute().to_dataframe()
series = series[(series.u_cov >= 0.5) & (series.r_cov >= 0.5)][["urban", "rural"]]
series["suhi"] = series.urban - series.rural
series["year"] = series.index.year
series.to_csv(f"{OUT}/suhi_per_scene.csv")

yearly = series.groupby("year")[["urban", "rural", "suhi"]].median()
slope, intercept, lo, hi = stats.theilslopes(yearly.suhi, yearly.index)
tau, p = stats.kendalltau(yearly.index, yearly.suhi)
u_slope = stats.theilslopes(yearly.urban, yearly.index)[0]

def period_mean(col, a, b):
    return yearly.loc[a:b, col].mean()
first, last = list(PERIODS.items())[0], list(PERIODS.items())[-1]
print(f"Scenes used: {len(series)} | years: {yearly.index.min()}–{yearly.index.max()}")
print(f"Mean SUHI {first[0]}: {period_mean('suhi', *first[1]):.1f} °C  |  {last[0]}: {period_mean('suhi', *last[1]):.1f} °C")
print(f"SUHI trend: {slope*10:+.2f} °C per decade (95 % CI {lo*10:+.2f} to {hi*10:+.2f}), Mann–Kendall p = {p:.3f}")
print(f"Urban surface temperature trend (weather-dependent): {u_slope*10:+.2f} °C per decade")

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.scatter(series.index, series.suhi, s=10, color="#bbbbbb", label="single scene")
ax.plot(pd.to_datetime(yearly.index.astype(str) + "-07-15"), yearly.suhi, "o-", color="#b2182b", lw=1.5, ms=4,
        label="summer median")
xs = np.array([yearly.index.min(), yearly.index.max()])
ax.plot(pd.to_datetime(xs.astype(str)), intercept + slope * xs, "--", color="k", lw=1,
        label=f"trend {slope*10:+.2f} °C/decade (" + ("p<0.01" if p < 0.01 else f"p={p:.2f}") + ")")
ax.axhline(0, color="k", lw=0.4)
ax.set(ylabel="Urban − rural LST (°C)", title="Surface urban heat island intensity in Brno, summer daytime (≈10:00)")
ax.legend(frameon=False, fontsize=8, ncol=3)
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_suhi_timeseries.png"); plt.show()

## 5b. Robustness: does the trend depend on the reference or on the scene dates?
A heat-island trend can appear without the city changing, if the **rural reference** changes: farmland at 10:00 is cool when crops are green (June, early July) and almost as hot as asphalt after the harvest (August) or in droughts (2018), and the early years use Landsat 5/7 while the recent ones use Landsat 8/9. Because clouds remove different days in different years, the share of August images also differs between periods. This section recomputes the trend with three references (farmland at the urban elevation, forest only, all non-urban land at the urban elevation), each with **July only** and with **June–July only** (before the harvest), shows the median day of year and the sensor mix of each period, and fits a regression on all scenes that compares each month only with itself (month-adjusted trend).

**How to read it.** June–July is the main result used in the report: it keeps 25 of 26 summers while always comparing the city with green fields. A single month leaves too many gaps (July has only one summer in 2000–2004), and including August mixes in harvested fields. The month-adjusted regression uses all scenes and should agree with the June–July result.

In [ ]:
# Does the trend depend on the rural reference or on the dates of the scenes?
u_all = lst.where(urban).median(("y", "x"))
REFS = {
    "Cropland & grass, same elevation (main)": rural,
    "Forest only": in_ring & (worldcover == 10),
    "All non-urban, same elevation": in_ring & worldcover.isin([10, 20, 30, 40]) & (abs(dem - z0) <= ELEV_TOL),
}

def trend_row(s):
    yr = s.groupby(s.index.year).median()
    if len(yr) < 8:
        return {"first_period": np.nan, "last_period": np.nan, "trend_per_decade": np.nan, "p": np.nan, "years": len(yr)}
    return {"first_period": yr.loc[first[1][0]:first[1][1]].mean(),
            "last_period": yr.loc[last[1][0]:last[1][1]].mean(),
            "trend_per_decade": stats.theilslopes(yr.values, yr.index)[0] * 10,
            "p": stats.kendalltau(yr.index, yr.values)[1], "years": len(yr)}

rows, ref_series = {}, {}
for name, m in REFS.items():
    s = (u_all - lst.where(m).median(("y", "x"))).compute().to_series().dropna()
    s = s[s.index.isin(series.index)]                     # same scenes as the main series
    ref_series[name] = s
    rows[name] = trend_row(s)
    rows[name + ", July only"] = trend_row(s[s.index.month == 7])
    rows[name + ", June–July"] = trend_row(s[s.index.month.isin([6, 7])])   # before the harvest: main result in the report
robust = pd.DataFrame(rows).T.round(2)
robust.to_csv(f"{OUT}/robustness_trend.csv")
display(robust)

# Season and sensor mix per period: if they differ a lot, part of the "trend" can come from them
info = pd.DataFrame({"doy": series.index.dayofyear,
                     "platform": inv.reindex(series.index.normalize())["platform"].values}, index=series.index)
info["period"] = pd.cut(info.index.year, [y[0] - 1 for y in PERIODS.values()] + [list(PERIODS.values())[-1][1]],
                        labels=list(PERIODS))
season = info.groupby("period", observed=False).agg(scenes=("doy", "size"), median_day_of_year=("doy", "median"))
season = season.join(info.groupby(["period", "platform"], observed=False).size().unstack(fill_value=0))
season.to_csv(f"{OUT}/robustness_season_sensor.csv")
display(season)

fig, ax = plt.subplots(figsize=(10, 3.4))
for (name, s), col in zip(ref_series.items(), ["#b2182b", "#2f6b3a", "#2a78d6"]):
    yr = s.groupby(s.index.year).median()
    ax.plot(yr.index, yr.values, "o-", ms=3, lw=1.4, color=col, label=name)
ax.axhline(0, color="k", lw=0.4)
ax.set(ylabel="Urban − rural LST (°C)", title="Heat island intensity with three rural references (summer medians)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig2b_robustness_references.png"); plt.show()

positive = (robust["trend_per_decade"] > 0).sum()
signif = (robust["p"] < 0.05).sum()
print(f"Positive trend in {positive} of {len(robust)} variants; significant (p < 0.05) in {signif}.")

# Trend adjusted for the month of each scene: OLS on all scenes with month dummies plus year.
# June scenes show a much larger heat island than August scenes (harvested, dry fields), so an uneven
# month mix between periods can create or hide a trend; the year coefficient is free of that effect.
def month_adjusted(s):
    s = s.dropna()
    X = pd.get_dummies(s.index.month, prefix="m").astype(float).values
    X = np.column_stack([X, s.index.year.values])
    b, *_ = np.linalg.lstsq(X, s.values, rcond=None)
    resid = s.values - X @ b
    dof = len(s) - X.shape[1]
    se = np.sqrt(resid @ resid / dof * np.linalg.inv(X.T @ X)[-1, -1])
    return {"trend_per_decade": b[-1] * 10, "se_per_decade": se * 10,
            "p": 2 * stats.t.sf(abs(b[-1] / se), dof), "scenes": len(s)}

u_s = u_all.compute().to_series()
u_s = u_s[u_s.index.isin(series.index)]
adj = {"Urban LST (absolute)": month_adjusted(u_s)}
for name, m in REFS.items():
    r_s = lst.where(m).median(("y", "x")).compute().to_series()
    r_s = r_s[r_s.index.isin(series.index)]
    adj[f"Rural LST (absolute): {name}"] = month_adjusted(r_s)
    adj[f"Heat island vs {name}"] = month_adjusted(ref_series[name])
month_adj = pd.DataFrame(adj).T.round(3)
month_adj.to_csv(f"{OUT}/robustness_month_adjusted.csv")
# absolute urban LST, June–July only (depends strongly on the weather of each image day)
u_jj = u_s[u_s.index.month.isin([6, 7])]
urban_jj = trend_row(u_jj)
print("Urban LST, June–July only:", {k: round(float(v), 2) for k, v in urban_jj.items()})
display(month_adj)
print("Median heat island by month (main reference):",
      series["suhi"].groupby(series.index.month).median().round(2).to_dict())


## 6. Where the city is hottest: maps by period
Each scene is expressed as an **anomaly**: pixel temperature minus that scene's rural median. The median anomaly per period shows where the city is persistently warmer than its surroundings, independent of the weather on the day.

In [ ]:
anom = lst - r_med                                          # °C above the rural reference
anom_period = {}
for name, (a, b) in PERIODS.items():
    sel = anom.sel(time=slice(f"{a}-01-01", f"{b}-12-31"))
    if sel.sizes["time"]:
        anom_period[name] = sel.median("time").compute()
names = list(anom_period)
p0, p1 = names[0], names[-1]
change = anom_period[p1] - anom_period[p0]

def show(ax, da, title, cmap, vmin, vmax, label):
    im = da.where(in_study).plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False)
    city.boundary.plot(ax=ax, color="k", lw=0.7)
    if districts is not None:
        districts.boundary.plot(ax=ax, color="k", lw=0.2, alpha=0.5)
    ax.set(title=title, xlabel="", ylabel="", xticks=[], yticks=[]); ax.set_aspect("equal")
    plt.colorbar(im, ax=ax, shrink=0.7, label=label)

fig, axs = plt.subplots(1, 3, figsize=(14, 4.6))
show(axs[0], anom_period[p0], f"Heat anomaly {p0}", "RdYlBu_r", -4, 10, "°C above rural")
show(axs[1], anom_period[p1], f"Heat anomaly {p1}", "RdYlBu_r", -4, 10, "°C above rural")
show(axs[2], change, f"Change {p0} → {p1}", "PuOr_r", -4, 4, "°C")
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_anomaly_maps.png"); plt.show()

## 7. Per-pixel trend of the heat anomaly
Summer median anomaly per year, then a least-squares slope per pixel (°C per decade). Pixels observed in fewer than 10 years are masked.

In [ ]:
yrs = sorted(set(pd.DatetimeIndex(anom.time.values).year))
annual = xr.concat([anom.sel(time=str(y)).median("time") for y in yrs], dim=pd.Index(yrs, name="year")).compute()
enough = annual.notnull().sum("year") >= 10
trend = (annual.polyfit("year", deg=1, skipna=True).polyfit_coefficients.sel(degree=1) * 10).where(enough)

fig, ax = plt.subplots(figsize=(6.5, 5.2))
show(ax, trend, f"Trend of the heat anomaly, {YEARS[0]}–{YEARS[-1]}", "PuOr_r", -2, 2, "°C per decade")
plt.tight_layout(); plt.savefig(f"{OUT}/fig4_pixel_trend.png"); plt.show()

## 8. Replicating Dobrovolný (2013) over 25 years
Dobrovolný (2013) studied Brno's surface heat island with **two** Landsat scenes (24 May 2001 and 15 June 2006). His method had three parts, and the next sections repeat each of them for every 5-year period from 2000 to 2026:

1. **Temperature by land-cover class** (his Table 1): how much warmer or cooler each class is than the mean of the study area. He found industrial and commercial areas 3–5 °C above the mean, and water and forests the coolest.
2. **SUHI from land-cover classes**: mean LST of urban classes minus mean LST of rural classes. He obtained **4.2 °C (2001)** and **6.7 °C (2006)**.
3. **Regressions on a 300 × 300 m grid**: NDVI explained **56–67 %** of LST variance and building density (DENS) **37–40 %**.

Two later studies help interpret the results: Geletič, Lehnert & Dobrovolný (2016) ranked Brno's **Local Climate Zones** by LST (heavy industry warmest, water and dense trees coolest), and Kirschner et al. (2024) found in Prague that NDVI explains 38–54 % of LST, with NDVI 0 → 1 cooling the surface by 10–15 °C.

**Difference from the original.** His classes came from his own object-based classification, which is not available. Here the six classes are rebuilt from OpenStreetMap land-use polygons, so values are comparable in order of magnitude, not identical. All temperatures are used as anomalies (pixel minus that scene's rural median), which removes the weather of each day; class deviations are unaffected by this choice, because the same offset is subtracted from every pixel of a scene.

### 8a. Six land-cover classes from OpenStreetMap
The classes follow Dobrovolný's Fig. 1c: (1) residential, (2) industry and transport, (3) city parks and recreation, (4) arable land and grassland, (5) forests and orchards, (6) water. Polygons are burned in the order arable, forest, residential, parks, industry, water, so parks inside residential blocks, industry and water win where polygons overlap. If the OpenStreetMap download fails, a coarser version is derived from ESA WorldCover (residential and industry cannot then be separated).

In [ ]:
CLASSES = {1: "Residential", 2: "Industry & transport", 3: "Parks & recreation",
           4: "Arable & grassland", 5: "Forest & orchards", 6: "Water"}
OSM_TAGS = {
    "landuse": ["residential", "industrial", "commercial", "retail", "railway", "construction",
                "farmland", "meadow", "farmyard", "vineyard", "grass", "allotments", "cemetery",
                "forest", "orchard", "reservoir", "basin"],
    "leisure": ["park", "garden", "pitch", "recreation_ground"],
    "natural": ["wood", "scrub", "grassland", "water"],
    "aeroway": ["aerodrome"],
}
def osm_class(row):
    lu, le, na, ae = (row.get(k) for k in ("landuse", "leisure", "natural", "aeroway"))
    if na == "water" or lu in ("reservoir", "basin"):                       return 6
    if lu in ("industrial", "commercial", "retail", "railway", "construction") or ae == "aerodrome": return 2
    if lu == "residential":                                                 return 1
    if le in ("park", "garden", "pitch", "recreation_ground") or lu in ("grass", "allotments", "cemetery"): return 3
    if lu in ("forest", "orchard") or na in ("wood", "scrub"):              return 5
    if lu in ("farmland", "meadow", "farmyard", "vineyard") or na == "grassland": return 4
    return 0

study_4326 = gpd.GeoSeries([study], crs=CRS).to_crs(4326).iloc[0]
try:
    osm = osm_features(ox.features_from_polygon, study_4326, OSM_TAGS)
    osm = osm[osm.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(CRS).copy()
    for k in ("landuse", "leisure", "natural", "aeroway"):
        if k not in osm.columns: osm[k] = None
    osm["cls"] = osm.apply(osm_class, axis=1)
    osm = osm[osm.cls > 0]
    burn_order = [4, 5, 1, 3, 2, 6]                      # later classes overwrite earlier ones (parks over residential blocks)
    shapes = [(g, c) for c in burn_order for g in osm.geometry[osm.cls == c]]
    lc = rasterize(shapes, out_shape=gbox.shape, transform=gbox.affine, fill=0).astype("uint8")
    landcover = xr.DataArray(lc, coords={"y": cube.y, "x": cube.x}, dims=("y", "x"))
    LC_SOURCE = "OpenStreetMap"
except Exception as e:
    print("OpenStreetMap land use failed, using WorldCover instead:", e)
    wc = worldcover
    landcover = xr.where(wc == 50, 1, xr.where(wc == 40, 4, xr.where(wc == 30, 4,
                xr.where(wc == 10, 5, xr.where(wc == 80, 6, 0))))).astype("uint8")
    LC_SOURCE = "ESA WorldCover (fallback)"
landcover = landcover.where(in_study, 0)
shares = pd.Series({CLASSES[c]: float((landcover == c).sum()) for c in CLASSES})
print("Land-cover source:", LC_SOURCE)
print((shares / shares.sum() * 100).round(1).astype(str) + " %")

cmap_lc = plt.matplotlib.colors.ListedColormap(["#e8e8e8", "#e6a07a", "#9e3d64", "#9ccf7c", "#f2e3a0", "#2f6b3a", "#5b9bd5"])
fig, ax = plt.subplots(figsize=(6, 5.5))
landcover.where(in_study).plot(ax=ax, cmap=cmap_lc, vmin=-0.5, vmax=6.5, add_colorbar=False)
city.boundary.plot(ax=ax, color="k", lw=0.7)
handles = [plt.matplotlib.patches.Patch(color=cmap_lc(c), label=n) for c, n in CLASSES.items()]
ax.legend(handles=handles, loc="lower left", fontsize=7, frameon=True)
ax.set(title=f"Land-cover classes ({LC_SOURCE})", xlabel="", ylabel="", xticks=[], yticks=[]); ax.set_aspect("equal")
plt.tight_layout(); plt.savefig(f"{OUT}/fig5_landcover_classes.png"); plt.show()

### 8b. Temperature by class and the class-based heat island (Dobrovolný's Table 1, for every period)
For each period: mean heat anomaly of each class **minus the mean of the whole study area**, exactly as in his Table 1. The class-based SUHI uses classes 1–2 as urban and 4–5 as rural, all elevations included, so it is directly comparable with his 4.2 and 6.7 °C (which came from single clear days in May and June; multi-summer medians are expected to be somewhat lower).

In [ ]:
# anomalies without the water mask, so the water class can be reported as in the original study
anom_all = cube["lst"] - r_med
anom_period_all = {name: anom_all.sel(time=slice(f"{a}-01-01", f"{b}-12-31")).median("time").compute()
                   for name, (a, b) in PERIODS.items() if name in anom_period}

valid_lc = in_study & (landcover > 0)
rows, suhi_cls = {}, {}
for name, da in anom_period_all.items():
    area_mean = float(da.where(valid_lc).mean())
    rows[name] = {CLASSES[c]: float(da.where(landcover == c).mean()) - area_mean for c in CLASSES}
    urban_m = float(da.where(landcover.isin([1, 2])).mean())
    rural_m = float(da.where(landcover.isin([4, 5])).mean())
    suhi_cls[name] = urban_m - rural_m
class_table = pd.DataFrame(rows).round(2)
class_table.loc["SUHI (classes 1–2 minus 4–5)"] = pd.Series(suhi_cls).round(2)
class_table.to_csv(f"{OUT}/class_deviation_by_period.csv")
display(class_table)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
colors_lc = {1: "#e6a07a", 2: "#9e3d64", 3: "#9ccf7c", 4: "#d8c46a", 5: "#2f6b3a", 6: "#5b9bd5"}
for c, n in CLASSES.items():
    ax.plot(class_table.columns, class_table.loc[n], "o-", color=colors_lc[c], label=n, lw=1.6, ms=4)
ax.axhline(0, color="k", lw=0.5)
ax.set(ylabel="°C vs study-area mean", title="Surface temperature deviation by land-cover class (after Dobrovolný 2013)")
ax.legend(frameon=False, fontsize=7, ncol=3, loc="upper left", bbox_to_anchor=(0, -0.12))
plt.tight_layout(); plt.savefig(f"{OUT}/fig6_class_deviation.png"); plt.show()

hot = class_table.drop(index="SUHI (classes 1–2 minus 4–5)")[p1].idxmax()
print(f"Warmest class in {p1}: {hot}, {class_table.loc[hot, p1]:+.1f} °C vs area mean "
      f"(was {class_table.loc[hot, p0]:+.1f} °C in {p0})")
print(f"Class-based SUHI: {suhi_cls[p0]:.1f} °C in {p0} → {suhi_cls[p1]:.1f} °C in {p1} "
      f"(Dobrovolný 2013: 4.2 °C on 24 May 2001, 6.7 °C on 15 June 2006)")

### 8c. Local Climate Zones (optional)
Local Climate Zones (Stewart & Oke 2012) describe urban form (compact or open, building height, industry, trees, water) and are the current standard for comparing urban climates. This cell reads the **global LCZ map** (Demuzere et al. 2022, ~100 m, nominal year 2018) directly from Zenodo, only for the Brno window. Geletič, Lehnert & Dobrovolný (2016) found LCZ 10 (heavy industry) the warmest zone in seven of eight Brno scenes, and LCZ G (water) and A (dense trees) the coolest. If the download fails, the notebook simply skips this part.

In [ ]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import reproject, Resampling

LCZ_URL = "/vsicurl/https://zenodo.org/records/8419340/files/lcz_filter_v3.tif"
LCZ_NAMES = {1: "1 Compact high-rise", 2: "2 Compact mid-rise", 3: "3 Compact low-rise", 4: "4 Open high-rise",
             5: "5 Open mid-rise", 6: "6 Open low-rise", 7: "7 Lightweight low-rise", 8: "8 Large low-rise",
             9: "9 Sparsely built", 10: "10 Heavy industry", 11: "A Dense trees", 12: "B Scattered trees",
             13: "C Bush, scrub", 14: "D Low plants", 15: "E Bare rock or paved", 16: "F Bare soil or sand",
             17: "G Water"}
LCZ_COLORS = {1: "#8c0000", 2: "#d10000", 3: "#ff0000", 4: "#bf4d00", 5: "#ff6600", 6: "#ff9955", 7: "#faee05",
              8: "#bcbcbc", 9: "#ffccaa", 10: "#555555", 11: "#006a00", 12: "#00aa00", 13: "#648525",
              14: "#b9db79", 15: "#000000", 16: "#fbf7ae", 17: "#6a6aff"}
lcz = None
try:
    with rasterio.Env(GDAL_HTTP_TIMEOUT="60", GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"):
        with rasterio.open(LCZ_URL) as src:
            win = from_bounds(*STUDY_BBOX, transform=src.transform)
            arr = src.read(1, window=win)
            tr = src.window_transform(win)
    dst = np.zeros(gbox.shape, "uint8")
    reproject(arr, dst, src_transform=tr, src_crs="EPSG:4326", dst_transform=gbox.affine, dst_crs=CRS,
              resampling=Resampling.nearest)
    lcz = xr.DataArray(dst, coords={"y": cube.y, "x": cube.x}, dims=("y", "x")).where(lambda a: a > 0)
except Exception as e:
    print("LCZ map could not be read, skipping this part:", e)

if lcz is not None:
    lcz_rows = {}
    for z, n in LCZ_NAMES.items():
        m = (lcz == z) & in_city
        if int(m.sum()) >= 20:                                  # ignore zones with very few pixels
            lcz_rows[n] = {p0: float(anom_period[p0].where(m).mean()), p1: float(anom_period[p1].where(m).mean()),
                           "pixels": int(m.sum()), "code": z}
    lcz_table = pd.DataFrame(lcz_rows).T.sort_values(p1)
    lcz_table.to_csv(f"{OUT}/lcz_anomaly.csv")

    fig, ax = plt.subplots(figsize=(6.5, 0.28 * len(lcz_table) + 1.2))
    ax.barh(lcz_table.index, lcz_table[p1], color=[LCZ_COLORS[int(c)] for c in lcz_table.code], edgecolor="k", lw=0.3)
    ax.scatter(lcz_table[p0], lcz_table.index, color="k", s=12, zorder=3, label=p0)
    ax.axvline(0, color="k", lw=0.5)
    ax.set(xlabel="°C above rural reference", title=f"Heat anomaly by Local Climate Zone in Brno (bars {p1})")
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT}/fig7_lcz_anomaly.png"); plt.show()
    print("Warmest LCZ:", lcz_table.index[-1], "| coolest:", lcz_table.index[0])

## 9. NDVI and building density on a 300 m grid
Following Dobrovolný (2013), the city is aggregated to **300 × 300 m cells**. For each cell: mean heat anomaly, mean NDVI, and **building density (DENS)**, the share of the cell covered by buildings. Building footprints come from OpenStreetMap (current state), so DENS is the same for both periods; this is a limitation for the early period, as new buildings would not have existed yet. Simple linear regressions (ordinary least squares, one predictor at a time) give the share of temperature variance explained (R²), compared with his 56–67 % (NDVI) and 37–40 % (DENS). The heat anomaly and NDVI of each cell are the **medians of all summer images (June–August) of the period**, averaged over the cell; water cells are excluded.

**Reading the result.** These regressions describe an **association**, not a measured causal effect: greener cells are also less built-up, Collection 2 emissivity is partly derived from vegetation, and neighbouring cells are not independent. The cell below therefore also fits NDVI and building density together and repeats the NDVI fit on a thinned 600 m grid.

In [ ]:
# NDVI composites for the first and last period
ndvi_period = {name: cube["ndvi"].sel(time=slice(f"{a}-01-01", f"{b}-12-31")).median("time").compute()
               for name, (a, b) in PERIODS.items() if name in (p0, p1)}

# building footprints -> 30 m binary raster -> share per 300 m cell
city_4326 = city.to_crs(4326).geometry.iloc[0]
try:
    bld = osm_features(ox.features_from_polygon, city_4326, {"building": True})
    bld = bld[bld.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(CRS)
    # burn footprints on a 10 m grid aligned with the cube, then average to the cube pixels (= built-up share)
    k = max(1, int(round(RES / 10)))
    a = gbox.affine
    fine_tr = rasterio.Affine(a.a / k, a.b, a.c, a.d, a.e / k, a.f)
    fine = rasterize([(g, 1) for g in bld.geometry], out_shape=(gbox.shape[0] * k, gbox.shape[1] * k),
                     transform=fine_tr, fill=0).astype("float32")
    built = fine.reshape(gbox.shape[0], k, gbox.shape[1], k).mean(axis=(1, 3))
    DENS_SOURCE = f"OpenStreetMap ({len(bld):,} buildings)"
except Exception as e:
    print("OpenStreetMap buildings failed, using WorldCover built-up share as a proxy:", e)
    built = (worldcover == 50).values.astype("uint8")
    DENS_SOURCE = "ESA WorldCover built-up (proxy)"
built = xr.DataArray(built.astype("float32"), coords={"y": cube.y, "x": cube.x}, dims=("y", "x"))
print("Building density source:", DENS_SOURCE)

F = max(1, int(round(300 / RES)))                      # pixels per 300 m cell side
def to_grid(da):
    return da.coarsen(x=F, y=F, boundary="trim").mean()

cell_city = to_grid(in_city.astype("float32")) > 0.5
cell_water = to_grid(water.astype("float32")) > 0.2
dens = to_grid(built)

reg_rows, grid_data = [], {}
for name in (p0, p1):
    g = xr.Dataset({"anom": to_grid(anom_period[name]), "ndvi": to_grid(ndvi_period[name]), "dens": dens})
    g = g.where(cell_city & ~cell_water).to_dataframe()[["anom", "ndvi", "dens"]].dropna()
    grid_data[name] = g
    f_n = stats.linregress(g.ndvi, g.anom)
    f_d = stats.linregress(g.dens, g.anom)
    reg_rows.append({"period": name, "cells": len(g),
                     "R2_NDVI": f_n.rvalue ** 2, "slope_per_0.1_NDVI_C": f_n.slope * 0.1,
                     "R2_DENS": f_d.rvalue ** 2, "slope_per_10pct_DENS_C": f_d.slope * 0.1})
reg_table = pd.DataFrame(reg_rows).set_index("period").round(2)
reg_table.to_csv(f"{OUT}/grid_regressions.csv")
display(reg_table)

# The NDVI and DENS fits above are SIMPLE regressions (one predictor each, ordinary least squares).
# They describe associations, not causal effects: greener cells are also less built-up (the two
# predictors are correlated), Collection 2 emissivity is partly derived from vegetation, and
# neighbouring cells are not independent (spatial autocorrelation), so R² and p-values look stronger
# than an experiment would show. Below: both predictors together, their correlation, and a check
# on a thinned grid (every second cell in x and y, 600 m spacing).
def ols(df, cols):
    X = np.column_stack([np.ones(len(df))] + [df[c].values for c in cols])
    b, *_ = np.linalg.lstsq(X, df.anom.values, rcond=None)
    r = df.anom.values - X @ b
    return b, 1 - (r @ r) / (((df.anom - df.anom.mean()) ** 2).sum())
multi_rows = []
for name, g in grid_data.items():
    b2, r2 = ols(g, ["ndvi", "dens"])
    xi = pd.Series(g.index.get_level_values("x")).rank(method="dense").values
    yi = pd.Series(g.index.get_level_values("y")).rank(method="dense").values
    thin = g[(xi % 2 == 0) & (yi % 2 == 0)]
    multi_rows.append({"period": name, "cells": len(g), "corr_NDVI_DENS": g.ndvi.corr(g.dens),
                       "R2_NDVI_plus_DENS": r2, "NDVI_per_0.1_C_holding_DENS": b2[1] * 0.1,
                       "DENS_per_10pct_C_holding_NDVI": b2[2] * 0.1,
                       "R2_NDVI_thinned_600m": stats.linregress(thin.ndvi, thin.anom).rvalue ** 2,
                       "cells_thinned": len(thin)})
multi_table = pd.DataFrame(multi_rows).set_index("period").round(2)
multi_table.to_csv(f"{OUT}/grid_regressions_multiple.csv")
display(multi_table)
print("Reference, Dobrovolný (2013), single dates 2001 and 2006: R² NDVI 0.56–0.67, R² DENS 0.37–0.40")

g = grid_data[p1]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, var, lab in ((axs[0], "ndvi", "NDVI"), (axs[1], "dens", "Building density (share of cell)")):
    f = stats.linregress(g[var], g.anom)
    ax.scatter(g[var], g.anom, s=5, alpha=0.35, color="#555555")
    xs = np.linspace(g[var].min(), g[var].max(), 10)
    ax.plot(xs, f.intercept + f.slope * xs, color="#b2182b", lw=1.6, label=f"R² = {f.rvalue**2:.2f}")
    ax.set(xlabel=lab, ylabel="°C above rural reference"); ax.legend(frameon=False)
fig.suptitle(f"300 m grid cells in Brno, {p1}", y=1.0)
plt.tight_layout(); plt.savefig(f"{OUT}/fig8_grid_regressions.png"); plt.show()

ndvi_fit = reg_table.loc[p1]
print(f"In {p1}, each +0.1 NDVI ≈ {float(ndvi_fit['slope_per_0.1_NDVI_C']):+.2f} °C "
      f"(Prague, Kirschner et al. 2024: about -1.0 to -1.5 °C per 0.1 NDVI)")

## 10. District ranking
Mean heat anomaly in the latest period, its change since the first period, the pixel trend, and the share of green cover (NDVI > 0.4). Districts that are both hot and warming fastest are the first candidates for adaptation.

In [ ]:
def zonal(da, geom):
    m = rasterize_geom(geom) & ~water
    return float(da.where(m).mean())

if districts is not None:
    districts["name"] = districts["name"].fillna("(no name)").astype(str)
    print(f"{len(districts)} districts, {int((districts['name'] == '(no name)').sum())} without a name")
    rows = []
    for _, d in districts.iterrows():
        rows.append({"district": d["name"],
                     f"anomaly_{p1}": zonal(anom_period[p1], d.geometry),
                     "change": zonal(change, d.geometry),
                     "trend_per_decade": zonal(trend, d.geometry),
                     "green_share": zonal((ndvi_period[p1] > 0.4).astype(float), d.geometry)})
    dist = pd.DataFrame(rows).dropna(subset=[f"anomaly_{p1}"])
    dist = dist.sort_values(f"anomaly_{p1}", ascending=False).round(2)
    dist.to_csv(f"{OUT}/district_ranking.csv", index=False)
    display(dist.head(10))

    top = dist.set_index("district")[f"anomaly_{p1}"].sort_values()
    fig, ax = plt.subplots(figsize=(6, 0.22 * len(top) + 1))
    ax.barh(top.index.astype(str), top.values, color=plt.cm.RdYlBu_r(np.clip((top.values + 4) / 14, 0, 1)))
    ax.set(xlabel="°C above rural reference", title=f"Heat anomaly by district, {p1}")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig9_district_ranking.png"); plt.show()
else:
    print("No district boundaries: skipping the ranking.")

## 11. Interactive dashboard
Writes `outputs/brno_heat_dashboard.html`, a single file you open in any web browser (internet needed for the base maps). It shows **three synchronised maps** for an early, a middle and a recent summer (by default the years closest to 2001, to the middle of the period and to the most recent summer that have at least 90 % of the city observed, skipping the striped Landsat 7 years 2012–13; set `SHOW_YEARS` to choose). Each map can show:

* **True colour**: Landsat red-green-blue from the clearest scenes of that summer, to see what was on the ground then;
* **Surface heat**: degrees above the rural surroundings (diverging blue-grey-red, grey = same as rural);
* **Vegetation**: NDVI (single-hue green);
* **Land cover**: built-up, sealed or bare; grass and crops; trees; water, **classified for each year from the same Landsat composites** with fixed spectral-index rules (NDVI and MNDWI), so the three years are directly comparable. No official land-cover map exists for these exact years at this detail. Sealed surfaces and bare soil are spectrally too similar to separate with simple indices, so they form one non-vegetated class; for surface heat, vegetated versus non-vegetated is the distinction that matters;
* **Heat change**: change since the first year (left map shows its true colour for reference).

Present-day high-resolution imagery (Esri) can be shown underneath with adjustable opacity. **Clicking** a place shows heat and NDVI for the three years, land cover in each year, today's land use, building cover, Local Climate Zone and district, with a short reading of what probably explains the pattern (became built-up, trees lost, densification, industrial surface, a cooling green area). A list of the 300 m cells that **warmed and cooled most** jumps straight to them. District boundaries are shown by default, with name and statistics on hover. Charts summarise the heat-island trend, temperature by land cover, the hottest districts and how much land changed cover (e.g. vegetation that became built-up).

In [ ]:
DASHBOARD_TEMPLATE = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Brno Surface Heat</title>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css">
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
  :root {
    --page: #f9f9f7; --surface: #fcfcfb; --surface-2: #f2f1ed; --border: #e2e0da;
    --ink: #0b0b0b; --ink-2: #52514e; --ink-3: #7a7974; --accent: #2a78d6;
    --warm: #d03b3b; --cool: #1c5cab; --radius: 10px;
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --page: #0d0d0d; --surface: #1a1a19; --surface-2: #242423; --border: #333331;
      --ink: #f0efec; --ink-2: #c3c2b7; --ink-3: #8f8e86; --accent: #3987e5;
      --warm: #e66767; --cool: #6da7ec;
    }
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--page); color: var(--ink);
         font: 14px/1.45 system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; }
  .wrap { max-width: 1500px; margin: 0 auto; padding: 20px 16px 40px; }
  header h1 { font-size: 22px; margin: 0 0 4px; letter-spacing: -0.01em; }
  header p { margin: 0; color: var(--ink-2); max-width: 900px; }
  .kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 10px; margin: 16px 0; }
  .kpi { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 10px 12px; }
  .kpi .l { color: var(--ink-3); font-size: 12px; }
  .kpi .v { font-size: 20px; font-weight: 600; margin-top: 2px; }
  .kpi .n { color: var(--ink-2); font-size: 12px; }
  .controls { display: flex; flex-wrap: wrap; gap: 10px 18px; align-items: center; background: var(--surface);
              border: 1px solid var(--border); border-radius: var(--radius); padding: 10px 12px; margin-bottom: 10px; }
  .seg { display: inline-flex; border: 1px solid var(--border); border-radius: 8px; overflow: hidden; }
  .seg button { border: 0; background: transparent; color: var(--ink-2); padding: 6px 12px; font: inherit; cursor: pointer; }
  .seg button + button { border-left: 1px solid var(--border); }
  .seg button.on { background: var(--accent); color: #fff; }
  label.tog { display: inline-flex; gap: 6px; align-items: center; color: var(--ink-2); cursor: pointer; }
  select { font: inherit; color: var(--ink); background: var(--surface); border: 1px solid var(--border); border-radius: 6px; padding: 4px 6px; }
  .legend { display: flex; align-items: center; gap: 8px; color: var(--ink-2); font-size: 12px; margin-left: auto; }
  .legend .bar { width: 180px; height: 10px; border-radius: 3px; border: 1px solid var(--border); }
  .legend .ticks { display: flex; justify-content: space-between; width: 180px; font-size: 11px; color: var(--ink-3); }
  .maps { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; }
  .mapcard { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden; }
  .mapcard .t { padding: 7px 10px; font-weight: 600; display: flex; justify-content: space-between; }
  .mapcard .t span { color: var(--ink-3); font-weight: 400; font-size: 12px; }
  .map { height: 52vh; min-height: 340px; background: var(--surface-2); }
  .row2 { display: grid; grid-template-columns: 1.3fr 1fr; gap: 10px; margin-top: 10px; }
  .card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 12px 14px; }
  .card h2 { font-size: 15px; margin: 0 0 8px; }
  .muted { color: var(--ink-3); }
  table.insp { border-collapse: collapse; width: 100%; font-variant-numeric: tabular-nums; }
  table.insp th, table.insp td { text-align: right; padding: 4px 8px; border-bottom: 1px solid var(--border); }
  table.insp th:first-child, table.insp td:first-child { text-align: left; color: var(--ink-2); }
  .hint { margin-top: 10px; padding: 9px 11px; border-radius: 8px; background: var(--surface-2); }
  .hint b { display: block; margin-bottom: 2px; }
  .facts { display: grid; grid-template-columns: auto 1fr; gap: 3px 12px; margin-top: 10px; color: var(--ink-2); }
  .facts div:nth-child(odd) { color: var(--ink-3); }
  .spots { display: grid; grid-template-columns: 1fr 1fr; gap: 6px 14px; }
  .spots h3 { font-size: 13px; margin: 2px 0 4px; color: var(--ink-2); }
  .spot { display: flex; justify-content: space-between; gap: 8px; width: 100%; text-align: left; border: 1px solid var(--border);
          background: transparent; color: var(--ink); border-radius: 7px; padding: 5px 8px; font: inherit; cursor: pointer; margin-bottom: 5px; }
  .spot:hover { border-color: var(--accent); }
  .spot .w { color: var(--warm); font-variant-numeric: tabular-nums; }
  .spot .c { color: var(--cool); font-variant-numeric: tabular-nums; }
  .charts { display: grid; grid-template-columns: repeat(2, 1fr); gap: 10px; margin-top: 10px; }
  .sw { display: inline-flex; align-items: center; gap: 4px; margin-right: 8px; white-space: nowrap; }
  .sw i { width: 11px; height: 11px; border-radius: 2px; display: inline-block; border: 1px solid rgba(0,0,0,.15); }
  .lu-tag { display: inline-flex; align-items: center; gap: 4px; }
  .lu-tag i { width: 9px; height: 9px; border-radius: 2px; display: inline-block; }
  path.district { filter: drop-shadow(0 0 1.2px rgba(0,0,0,.85)); }
  .leaflet-tooltip.dtip { font: 12px/1.35 system-ui, sans-serif; padding: 6px 8px; }
  .charts canvas { width: 100% !important; height: 240px !important; }
  footer { margin-top: 18px; color: var(--ink-3); font-size: 12px; }
  .leaflet-container { font: inherit; }
  @media (max-width: 1000px) {
    .maps, .charts, .row2, .spots { grid-template-columns: 1fr; }
    .map { height: 44vh; }
    .legend { margin-left: 0; }
  }
  .yearbar { display: flex; flex-wrap: wrap; gap: 10px 14px; align-items: center; margin-bottom: 8px; }
  .yearbar input[type=range] { flex: 1; min-width: 200px; }
  .yearbar .play { border: 1px solid var(--border); background: var(--surface); color: var(--ink); border-radius: 8px; padding: 5px 12px; font: inherit; cursor: pointer; }
  #ylabel { font-variant-numeric: tabular-nums; min-width: 260px; }
  #mapY { height: 60vh; min-height: 360px; background: var(--surface-2); border-radius: 8px; }
</style>
</head>
<body>
<div class="wrap">
  <header>
    <h1>Surface heat in Brno, 2000–2026</h1>
    <p>Landsat summer surface temperature (≈10:00 local time) expressed as degrees above the rural surroundings, with vegetation (NDVI), land cover and true-colour imagery for three years. Click any place on a map to compare the three years and see what may explain the change.</p>
  </header>
  <div class="kpis" id="kpis"></div>

  <div class="controls">
    <div class="seg" id="mode">
      <button data-m="rgb">True colour</button>
      <button data-m="heat" class="on">Surface heat</button>
      <button data-m="ndvi">Vegetation</button>
      <button data-m="lu">Land cover</button>
      <button data-m="change">Heat change</button>
    </div>
    <label class="tog">Basemap
      <select id="basemap">
        <option value="cuzk">Orthophoto ČR, ČÚZK (~25 cm)</option>
        <option value="esri">Esri World Imagery</option>
        <option value="maptiler">MapTiler Satellite</option>
        <option value="light">Light street map</option>
      </select></label>
    <label class="tog">Opacity <input type="range" id="opacity" min="0" max="100" value="75"></label>
    <label class="tog"><input type="checkbox" id="showDistricts" checked> Districts</label>
    <div class="legend" id="legend"></div>
  </div>

  <div class="maps" id="maps"></div>

  <div class="row2">
    <div class="card" id="inspector">
      <h2>Inspector</h2>
      <p class="muted" id="inspEmpty">Click a place on any map, or pick one from the list.</p>
      <div id="inspBody" hidden></div>
    </div>
    <div class="card">
      <h2>Places to inspect</h2>
      <p class="muted" style="margin-top:-4px">300 m cells inside the city with the largest change in heat between the first and the last year.</p>
      <div class="spots">
        <div><h3>Warmed most</h3><div id="spotsWarm"></div></div>
        <div><h3>Cooled most</h3><div id="spotsCool"></div></div>
      </div>
    </div>
  </div>

  <div class="card" style="margin-top:10px">
    <h2>Year by year, 2000–2026</h2>
    <div class="yearbar">
      <div class="seg" id="ymode"><button data-m="heat" class="on">Surface heat</button><button data-m="rgb">True colour</button></div>
      <button class="play" id="yplay">▶ Play</button>
      <input type="range" id="yslider" min="0" max="0" step="1" value="0" aria-label="Year">
      <span id="ylabel"></span>
      <div class="legend" id="ylegend" style="margin-left:0"></div>
    </div>
    <div id="mapY"></div>
    <p class="muted" style="margin:6px 0 0;font-size:12px">One map per summer: median heat of the clear June–July images of that year, before the harvest, in °C above the countryside. Years without a clear June–July image use their August images and are marked; single images are noisier than multi-year medians, so compare patterns rather than single values.</p>
  </div>

  <div class="charts">
    <div class="card"><h2>Heat island by month (urban minus farmland)</h2><canvas id="cSuhi"></canvas><p class="muted" style="margin:6px 0 0;font-size:12px">Compare years within the same month: the heat island is large in June and July and near zero in August, when harvested fields heat up.</p></div>
    <div class="card"><h2>Temperature by land cover</h2><canvas id="cClass"></canvas></div>
    <div class="card"><h2>Hottest districts, latest period</h2><canvas id="cDist"></canvas></div>
    <div class="card"><h2>Land cover in the city (km²)</h2><canvas id="cLU"></canvas><p class="muted" id="luNote" style="margin:6px 0 0"></p></div>
  </div>

  <footer>
    Data: Landsat Collection 2 Level-2 surface temperature (USGS/NASA) via Microsoft Planetary Computer; land use and buildings © OpenStreetMap contributors; Local Climate Zones: Demuzere et al. (2022); basemaps: orthophoto © ČÚZK, imagery © Esri, Maxar, Earthstar Geographics, © MapTiler, © CARTO. Method after Dobrovolný (2013), Theor. Appl. Climatol. 112: 89–98. Basemaps show the present day, not the year of each map. Land cover per year is classified from the same Landsat composites with spectral indices (NDVI for vegetation, MNDWI for water) and fixed thresholds; sealed surfaces and bare soil cannot be told apart this way and form one class, so check doubtful places against the true-colour image.
  </footer>
</div>

<script>
const D = __PAYLOAD__;
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();

// ---------- decode the value grids ----------
function b64(s) { const b = atob(s), u = new Uint8Array(b.length); for (let i = 0; i < b.length; i++) u[i] = b.charCodeAt(i); return u; }
const G = {};
for (const [k, v] of Object.entries(D.grids)) {
  if (!v) continue;
  const u = b64(v);
  const categorical = k === 'lc' || k === 'lcz' || k === 'district' || k.startsWith('lu_');
  G[k] = categorical ? u : new Int16Array(u.buffer);
}
function val(name, i) {
  const a = G[name]; if (!a || i < 0) return null;
  const v = a[i];
  if (a instanceof Int16Array) return v === -32768 ? null : v / 100;
  return v === 0 ? null : v;
}

// ---------- KPI tiles ----------
document.getElementById('kpis').innerHTML = D.kpis.map(k =>
  `<div class="kpi"><div class="l">${k.label}</div><div class="v">${k.value}</div>${k.note ? `<div class="n">${k.note}</div>` : ''}</div>`).join('');

// ---------- maps ----------
const Y = D.years, maps = [], overlays = [], markers = [], distLayers = [];
// basemaps: for visual context only (present day), never used in the analysis
const BASEMAPS = {
  cuzk: () => L.tileLayer.wms('https://ags.cuzk.cz/arcgis1/services/ORTOFOTO/MapServer/WMSServer',
          { layers: '0', format: 'image/jpeg', transparent: false, version: '1.3.0', maxZoom: 20,
            attribution: 'Orthophoto © ČÚZK' }),
  esri: () => L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
          { maxZoom: 19, attribution: 'Imagery © Esri, Maxar, Earthstar Geographics' }),
  maptiler: () => L.tileLayer(`https://api.maptiler.com/maps/satellite/256/{z}/{x}/{y}.jpg?key=${D.maptilerKey}`,
          { maxZoom: 20, attribution: '© MapTiler © OpenStreetMap contributors' }),
  light: () => L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
          { maxZoom: 20, attribution: '© OpenStreetMap contributors © CARTO' }),
};
if (!D.maptilerKey) document.querySelector('#basemap option[value="maptiler"]').remove();
const base = [];
const mapsEl = document.getElementById('maps');
Y.forEach((y, i) => {
  const card = document.createElement('div'); card.className = 'mapcard';
  const sub = i === 0 ? 'early' : i === 1 ? 'middle' : 'recent';
  card.innerHTML = `<div class="t">${y}<span>${sub}</span></div><div class="map" id="map${i}"></div>`;
  mapsEl.appendChild(card);
  const m = L.map(`map${i}`, { zoomControl: i === 0, attributionControl: i === 2, minZoom: 10, maxZoom: 19 })
             .setView(D.center, 12);
  base.push(BASEMAPS.cuzk().addTo(m)); maps.push(m);
});

// keep the three maps in step
let syncing = false;
maps.forEach((m, i) => m.on('move', () => {
  if (syncing) return; syncing = true;
  maps.forEach((o, j) => { if (j !== i) o.setView(m.getCenter(), m.getZoom(), { animate: false }); });
  syncing = false;
}));

let mode = 'heat';
function layerFor(i) {
  const y = String(Y[i]);
  if (mode === 'change') return i === 0 ? D.layers[y].rgb : D.layers[y].change;
  if (mode === 'lu') return D.layers[y].lu;
  return D.layers[y][mode];
}
function drawOverlays() {
  const op = document.getElementById('opacity').value / 100;
  maps.forEach((m, i) => {
    if (overlays[i]) m.removeLayer(overlays[i]);
    overlays[i] = L.imageOverlay(layerFor(i), D.bounds, { opacity: (mode === 'rgb' || (mode === 'change' && i === 0)) ? 1 : op,
                                                           interactive: false }).addTo(m);
    if (distLayers[i]) distLayers[i].bringToFront();
  });
  drawLegend();
}
function gradient(stops) { return `linear-gradient(90deg, ${stops.join(',')})`; }
function drawLegend() {
  const el = document.getElementById('legend');
  if (mode === 'rgb') { el.innerHTML = '<span>Landsat true colour (30 m), clearest scenes of each summer</span>'; return; }
  if (mode === 'lu') { el.innerHTML = Object.entries(D.luClasses).map(([c, n]) =>
      `<span class="sw"><i style="background:${D.luColors[c]}"></i>${n}</span>`).join(''); return; }
  const cfg = {
    heat:   { stops: D.heatStops, ticks: ['−4', '0', '+5', '+10 °C'], label: 'Above rural surroundings' },
    change: { stops: D.heatStops, ticks: ['−4', '0', '', '+4 °C'], label: `Change since ${Y[0]} (left: ${Y[0]} true colour)` },
    ndvi:   { stops: D.ndviStops, ticks: ['0', '0.4', '', '0.8'], label: 'NDVI' },
  }[mode];
  el.innerHTML = `<span>${cfg.label}</span><div><div class="bar" style="background:${gradient(cfg.stops)}"></div>
    <div class="ticks">${cfg.ticks.map(t => `<span>${t}</span>`).join('')}</div></div>`;
}
document.querySelectorAll('#mode button').forEach(b => b.onclick = () => {
  document.querySelectorAll('#mode button').forEach(x => x.classList.toggle('on', x === b));
  mode = b.dataset.m; drawOverlays();
});
document.getElementById('opacity').oninput = drawOverlays;
document.getElementById('basemap').onchange = e => maps.forEach((m, i) => {
  m.removeLayer(base[i]); base[i] = BASEMAPS[e.target.value]().addTo(m); base[i].bringToBack();
});

// districts: shown by default; the transparent fill makes the whole area respond to hover
const dStyle = { color: '#ffffff', weight: 1.4, opacity: 0.95, fill: true, fillColor: '#ffffff', fillOpacity: 0, className: 'district' };
if (D.districts) {
  maps.forEach((m, i) => {
    distLayers[i] = L.geoJSON(D.districts, {
      style: dStyle,
      onEachFeature: (f, l) => {
        const pr = f.properties, sgn = v => (v >= 0 ? '+' : '') + v.toFixed(1);
        l.bindTooltip(`<b>${pr.name}</b>` +
          (pr.anomaly == null ? '' : `<br>${sgn(pr.anomaly)} °C above rural`) +
          (pr.change == null ? '' : `<br>Change since first period: ${sgn(pr.change)} °C`) +
          (pr.green_share == null ? '' : `<br>Green cover: ${(pr.green_share * 100).toFixed(0)} %`),
          { sticky: true, direction: 'top', offset: [0, -8], className: 'dtip' });
        l.on('mouseover', () => l.setStyle({ weight: 3, fillOpacity: 0.08 }));
        l.on('mouseout', () => l.setStyle({ weight: dStyle.weight, fillOpacity: 0 }));
      }
    }).addTo(m);
  });
} else { document.getElementById('showDistricts').parentElement.hidden = true; }
document.getElementById('showDistricts').onchange = e => maps.forEach((m, i) => {
  if (!distLayers[i]) return; e.target.checked ? distLayers[i].addTo(m) : m.removeLayer(distLayers[i]);
});

// ---------- inspector ----------
function cellIndex(latlng) {
  const p = L.CRS.EPSG3857.project(latlng);
  const c = Math.floor((p.x - D.grid.x0) / D.grid.res), r = Math.floor((D.grid.y0 - p.y) / D.grid.res);
  if (c < 0 || r < 0 || c >= D.grid.w || r >= D.grid.h) return -1;
  return r * D.grid.w + c;
}
const fmt = (v, d = 1, s = '') => v == null ? '–' : (v > 0 && s ? '+' : '') + v.toFixed(d) + s;
const LU_SHORT = { 1: 'Built/bare', 3: 'Grass', 4: 'Trees', 5: 'Water' };
function luShort(c) { return c ? `<span class="lu-tag"><i style="background:${D.luColors[c]}"></i>${LU_SHORT[c] || D.luClasses[c]}</span>` : '–'; }
function luTag(c) {
  return c ? `<span class="lu-tag"><i style="background:${D.luColors[c]}"></i>${D.luClasses[c]}</span>` : '–';
}
function explain(h, n, dens, lc, lu) {
  const dH = (h[2] != null && h[0] != null) ? h[2] - h[0] : null;
  const dN = (n[2] != null && n[0] != null) ? n[2] - n[0] : null;
  const cls = lc ? D.classes[lc] : null;
  if (dH == null) return ['Not enough clear observations here', 'Clouds or data gaps hide this place in at least one of the years.'];
  const l0 = lu && lu[0], l2 = lu && lu[2], nm = c => D.luClasses[c].toLowerCase();
  if ((l0 === 3 || l0 === 4) && l2 === 1)
    return ['Vegetation replaced by built-up or bare ground', `This place was ${nm(l0)} in ${Y[0]} and is built-up, sealed or bare in ${Y[2]}; the surface warmed ${fmt(dH, 1, ' °C')}. Compare the true-colour images to see new buildings, paving or a construction site.`];
  if (l0 === 4 && l2 === 3)
    return ['Trees lost', `Dense vegetation in ${Y[0]} became ${nm(l2)} by ${Y[2]} (heat ${fmt(dH, 1, ' °C')}). Felled trees or cleared land; replanting would restore the cooling.`];
  if ((l0 === 1 || l0 === 3) && l2 === 4 && dH <= 0)
    return ['Trees grew here', `Was ${nm(l0)} in ${Y[0]}, dense vegetation in ${Y[2]}; the surface cooled ${fmt(dH, 1, ' °C')}.`];
  if (dN != null && dN <= -0.1 && dH >= 0.5)
    return ['Vegetation lost, surface warmed', `NDVI fell by ${(-dN).toFixed(2)} and the surface warmed ${fmt(dH, 1, ' °C')} since ${Y[0]}. This usually means new buildings, paving or tree removal. Compare the true-colour images of ${Y[0]} and ${Y[2]}.`];
  if (dN != null && dN >= 0.1 && dH <= -0.3)
    return ['Greener and cooler', `NDVI rose by ${dN.toFixed(2)} and the surface cooled ${fmt(dH, 1, ' °C')}: vegetation growth, a new park or maturing trees.`];
  if (cls === 'Industry & transport')
    return ['Industrial or transport surface', `Large roofs, paved yards and rail or road areas store and re-emit heat (${fmt(h[2], 1, ' °C')} above rural). Typical measures: cool or green roofs, unsealing paved areas, trees along the edges.`];
  if (dens != null && dens >= 0.35 && n[2] != null && n[2] < 0.3)
    return ['Dense built-up area with little greenery', `About ${(dens * 100).toFixed(0)} % of the area is covered by buildings and NDVI is only ${n[2].toFixed(2)}. Street trees, courtyard greening and green roofs give the most cooling here.`];
  if (h[2] != null && h[2] < 0)
    return ['A cool place', `${cls ? cls + ': ' : ''}cooler than the rural surroundings (${fmt(h[2], 1, ' °C')}). Areas like this cool their neighbourhood and are worth protecting.`];
  if (Math.abs(dH) < 0.5)
    return ['Little change', `Heat changed by only ${fmt(dH, 1, ' °C')} since ${Y[0]}; the level mainly reflects the type of surface${cls ? ' (' + cls.toLowerCase() + ')' : ''}.`];
  const veg = dN == null ? 'vegetation unknown' : Math.abs(dN) < 0.03 ? 'vegetation unchanged' : `NDVI ${dN > 0 ? 'up' : 'down'} ${Math.abs(dN).toFixed(2)}`;
  return [dH > 0 ? 'Warmer than before' : 'Cooler than before', `Change of ${fmt(dH, 1, ' °C')} since ${Y[0]}, ${veg}. Check the true-colour images for new buildings, paving or vegetation changes; with vegetation unchanged, the change may come from surface materials or from the wider warming of the city.`];
}
function inspect(latlng) {
  const i = cellIndex(latlng);
  markers.forEach(mk => mk.remove()); markers.length = 0;
  maps.forEach(m => markers.push(L.circleMarker(latlng, { radius: 7, color: '#fff', weight: 2, fillColor: '#111', fillOpacity: 0.9 }).addTo(m)));
  const h = Y.map(y => val('heat_' + y, i)), n = Y.map(y => val('ndvi_' + y, i));
  const dens = val('dens', i), lc = val('lc', i), lcz = val('lcz', i), di = val('district', i);
  const lu = Y.map(y => val('lu_' + y, i));
  const [title, text] = explain(h, n, dens, lc, lu);
  document.getElementById('inspEmpty').hidden = true;
  const body = document.getElementById('inspBody'); body.hidden = false;
  body.innerHTML = `
    <table class="insp"><thead><tr><th></th>${Y.map(y => `<th>${y}</th>`).join('')}<th>Change</th></tr></thead><tbody>
      <tr><td>Heat above rural (°C)</td>${h.map(v => `<td>${fmt(v, 1)}</td>`).join('')}<td>${h[2] != null && h[0] != null ? fmt(h[2] - h[0], 1, ' °C') : '–'}</td></tr>
      <tr><td>Vegetation (NDVI)</td>${n.map(v => `<td>${fmt(v, 2)}</td>`).join('')}<td>${n[2] != null && n[0] != null ? fmt(n[2] - n[0], 2, ' ') : '–'}</td></tr>
      <tr><td>Land cover</td>${lu.map(v => `<td>${luTag(v)}</td>`).join('')}<td>${lu[0] && lu[2] ? (lu[0] === lu[2] ? 'same' : 'changed') : '–'}</td></tr>
    </tbody></table>
    <div class="hint"><b>${title}</b>${text}</div>
    <div class="facts">
      <div>Land use today (OSM)</div><div>${lc ? D.classes[lc] : '–'}</div>
      <div>Building cover</div><div>${dens == null ? '–' : (dens * 100).toFixed(0) + ' % of the area'}</div>
      ${G.lcz ? `<div>Local Climate Zone</div><div>${lcz ? D.lcz[lcz] : '–'}</div>` : ''}
      ${G.district ? `<div>District</div><div>${di ? D.districtNames[di - 1] : 'outside the city'}</div>` : ''}
      <div>Location</div><div>${latlng.lat.toFixed(5)}, ${latlng.lng.toFixed(5)}</div>
    </div>`;
}
maps.forEach(m => m.on('click', e => inspect(e.latlng)));

function spotList(list, el, cls) {
  document.getElementById(el).innerHTML = list.map((s, k) =>
    `<button class="spot" data-k="${k}"><span>#${k + 1} · ${s.lu0 && s.lu1 && s.lu0 !== s.lu1 ? `${luShort(s.lu0)} → ${luShort(s.lu1)}` : `NDVI ${s.dndvi >= 0 ? '+' : ''}${s.dndvi.toFixed(2)}`}</span><span class="${cls}">${s.dheat >= 0 ? '+' : ''}${s.dheat.toFixed(1)} °C</span></button>`).join('')
    || '<p class="muted">None found.</p>';
  document.querySelectorAll(`#${el} .spot`).forEach(b => b.onclick = () => {
    const s = list[+b.dataset.k], ll = L.latLng(s.lat, s.lon);
    maps[0].setView(ll, 16); inspect(ll);
    document.getElementById('maps').scrollIntoView({ behavior: 'smooth', block: 'start' });
  });
}
spotList(D.hotspots.warming, 'spotsWarm', 'w');
spotList(D.hotspots.cooling, 'spotsCool', 'c');
drawOverlays();

// ---------- charts ----------
const ink2 = css('--ink-2'), grid = css('--border');
Chart.defaults.color = ink2; Chart.defaults.font.family = 'system-ui, -apple-system, "Segoe UI", Roboto, sans-serif';
const axes = { x: { grid: { color: grid } }, y: { grid: { color: grid } } };
new Chart(document.getElementById('cSuhi'), {
  type: 'line',
  data: { labels: D.charts.years, datasets: Object.entries(D.charts.suhiMonth).map(([m, v], k) => (
    { label: m, data: v, borderColor: ['#2a78d6', '#eb6834', '#1baf7a'][k], backgroundColor: ['#2a78d6', '#eb6834', '#1baf7a'][k],
      borderWidth: 2, pointRadius: 3, spanGaps: true })) },
  options: { maintainAspectRatio: false, interaction: { mode: 'index', intersect: false },
    plugins: { legend: { labels: { boxWidth: 12 } }, tooltip: { callbacks: { label: c => c.parsed.y == null ? '' : `${c.dataset.label}: ${c.parsed.y.toFixed(2)} °C` } } },
    scales: { ...axes, y: { ...axes.y, title: { display: true, text: 'Urban − rural (°C)' } } } }
});
const CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#008300', '#6250d6'];
new Chart(document.getElementById('cClass'), {
  type: 'line',
  data: { labels: D.charts.periods, datasets: Object.entries(D.charts.classes).map(([n, v], k) =>
    ({ label: n, data: v, borderColor: CAT[k], backgroundColor: CAT[k], borderWidth: 2, pointRadius: 3, spanGaps: true })) },
  options: { maintainAspectRatio: false, interaction: { mode: 'index', intersect: false },
    plugins: { legend: { position: 'bottom', labels: { boxWidth: 10, font: { size: 11 } } },
               tooltip: { callbacks: { label: c => `${c.dataset.label}: ${c.parsed.y >= 0 ? '+' : ''}${c.parsed.y.toFixed(1)} °C` } } },
    scales: { ...axes, y: { ...axes.y, title: { display: true, text: '°C vs study-area mean' } } } }
});
if (D.charts.districts.length) {
  const ds = D.charts.districts.slice().reverse();
  new Chart(document.getElementById('cDist'), {
    type: 'bar',
    data: { labels: ds.map(d => d.name), datasets: [{ label: '°C above rural', data: ds.map(d => d.value),
      backgroundColor: '#ec7a6e', borderRadius: 4, borderSkipped: 'start', barPercentage: 0.75 }] },
    options: { indexAxis: 'y', maintainAspectRatio: false, plugins: { legend: { display: false },
      tooltip: { callbacks: { label: c => `${c.parsed.x.toFixed(1)} °C above rural` } } },
      scales: { x: { grid: { color: grid }, title: { display: true, text: '°C above rural' } }, y: { grid: { display: false }, ticks: { autoSkip: false, font: { size: 11 } } } } }
  });
} else { document.getElementById('cDist').parentElement.innerHTML += '<p class="muted">No district boundaries available.</p>'; }
const YEAR_SHADES = ['#9ec5f4', '#3987e5', '#104281'];
new Chart(document.getElementById('cLU'), {
  type: 'bar',
  data: { labels: Object.keys(D.luStats), datasets: Y.map((y, k) => ({ label: String(y), data: Object.values(D.luStats).map(v => v[k]),
    backgroundColor: YEAR_SHADES[k], borderRadius: 4, borderSkipped: 'start', categoryPercentage: 0.75, barPercentage: 0.9 })) },
  options: { maintainAspectRatio: false, plugins: { legend: { labels: { boxWidth: 12 } },
    tooltip: { callbacks: { label: c => `${c.dataset.label}: ${c.parsed.y.toFixed(1)} km²` } } },
    scales: { x: { grid: { display: false }, ticks: { font: { size: 11 } } }, y: { grid: { color: grid }, title: { display: true, text: 'km² inside the city' } } } }
});
document.getElementById('luNote').textContent =
  `${Y[0]}→${Y[2]}: ${D.transitions.veg_to_built.toFixed(1)} km² of vegetation became built-up; ` +
  `trees lost ${D.transitions.trees_lost.toFixed(1)} km², gained ${D.transitions.trees_gained.toFixed(1)} km².`;

// ---------- year-by-year viewer ----------
if (D.annual && D.annual.years && D.annual.years.length) {
  const A = D.annual, n = A.years.length;
  const mY = L.map('mapY', { minZoom: 10, maxZoom: 19 }).setView(D.center, 12);
  BASEMAPS.cuzk().addTo(mY);
  if (D.districts) L.geoJSON(D.districts, { interactive: false,
      style: { color: '#ffffff', weight: 1.2, opacity: 0.9, fill: false } }).addTo(mY);
  let yMode = 'heat', yIdx = n - 1, yOv = null, timer = null, preloaded = false;
  const sl = document.getElementById('yslider'), lab = document.getElementById('ylabel'), btn = document.getElementById('yplay');
  sl.max = n - 1; sl.value = yIdx;
  document.getElementById('ylegend').innerHTML = `<span>Above countryside</span><div><div class="bar" style="background:${gradient(D.heatStops)}"></div>
    <div class="ticks"><span>−4</span><span>0</span><span>+5</span><span>+10 °C</span></div></div>`;
  if (!A.rgb.length) document.querySelector('#ymode button[data-m="rgb"]').remove();
  const urlFor = y => A.dir + (yMode === 'rgb' ? `rgb_${y}.jpg` : `heat_${y}.png`);
  function preload() { if (preloaded) return; preloaded = true;
    A.years.forEach(y => { new Image().src = A.dir + `heat_${y}.png`; if (A.rgb.includes(y)) new Image().src = A.dir + `rgb_${y}.jpg`; }); }
  function drawY() {
    const y = A.years[yIdx];
    if (yOv) { mY.removeLayer(yOv); yOv = null; }
    const missing = yMode === 'rgb' && !A.rgb.includes(y);
    if (!missing) {
      const op = document.getElementById('opacity').value / 100;
      yOv = L.imageOverlay(urlFor(y), D.bounds, { opacity: yMode === 'rgb' ? 1 : op, interactive: false }).addTo(mY);
    }
    document.getElementById('ylegend').style.visibility = yMode === 'rgb' ? 'hidden' : 'visible';
    const flag = A.basis[y] && A.basis[y] !== 'June–July' ? ` · <span style="color:var(--warm)">${A.basis[y]}</span>` : ' · June–July';
    lab.innerHTML = `<b>${y}</b>${flag} · ${A.n[y]} image${A.n[y] === 1 ? '' : 's'}${missing ? ' · no true-colour image' : ''}`;
  }
  sl.oninput = () => { yIdx = +sl.value; drawY(); };
  document.querySelectorAll('#ymode button').forEach(b => b.onclick = () => {
    document.querySelectorAll('#ymode button').forEach(x => x.classList.toggle('on', x === b));
    yMode = b.dataset.m; preloaded = false; drawY(); });
  btn.onclick = () => {
    if (timer) { clearInterval(timer); timer = null; btn.textContent = '▶ Play'; return; }
    preload(); btn.textContent = '❚❚ Pause';
    timer = setInterval(() => { yIdx = (yIdx + 1) % n; sl.value = yIdx; drawY(); }, 900);
  };
  document.getElementById('opacity').addEventListener('input', drawY);
  drawY();
}
</script>
</body>
</html>
'''

In [ ]:
# Builds outputs/brno_heat_dashboard.html: three synchronised maps (early, middle, recent year),
# each switchable between Landsat true colour, surface heat and vegetation (NDVI), over
# high-resolution imagery, with a click inspector, hotspot list and summary charts.
import base64, io, json
from PIL import Image
from pyproj import Transformer
from odc.geo.geobox import GeoBox
from odc.geo.xr import assign_crs
import matplotlib.colors as mcolors

SHOW_YEARS = None          # e.g. (2001, 2014, 2026); None = chosen automatically (see below)
import os
# Optional MapTiler satellite basemap. The key is read from an environment variable so it is never
# committed with the notebook. NOTE: it is still written into the published HTML (browsers need it),
# so restrict it to your GitHub Pages domain in MapTiler Cloud. Leave it unset to use ČÚZK / Esri only.
MAPTILER_KEY = os.environ.get("MAPTILER_KEY", "")
DASH_BUFFER_M = 1500       # area shown: city boundary plus this buffer
N_HOTSPOTS = 10            # places listed for inspection (warming and cooling)
ANNUAL_RGB = False         # True: year-by-year viewer also gets a true-colour image per summer (27 extra downloads, slow)
JJ = (6, 7)                # months used for the yearly heat maps (before the harvest, as in the report)

# ---------- 0. yearly heat maps from June–July images (all summer images if a year has none) ---
t_all = pd.DatetimeIndex(anom.time.values)
YEARS_ALL = sorted(set(t_all.year))
BASIS, NIMG, yearly_maps = {}, {}, []
for y in YEARS_ALL:
    sel = (t_all.year == y) & t_all.month.isin(JJ)
    BASIS[y] = "June–July"
    if sel.sum() == 0:                                    # e.g. a summer with clear images only in August
        sel = t_all.year == y
        BASIS[y] = "August only (no clear June–July image)"
    NIMG[y] = int(sel.sum())
    yearly_maps.append(anom.isel(time=np.where(sel)[0]).median("time"))
annual_jj = xr.concat(yearly_maps, dim=pd.Index(YEARS_ALL, name="year")).compute()
print("Yearly heat maps:", {y: f"{NIMG[y]} img, {BASIS[y].split(' ')[0]}" for y in YEARS_ALL})

# ---------- 1. pick the three years -------------------------------------------------------
cov = (annual_jj.where(in_city).notnull().sum(("y", "x")) / int(in_city.sum())).to_series()
MIN_COVER = 0.9
def nearest(target, exclude=(), prefer_recent=False):
    # year closest to the target with at least 90 % of the city observed (else the best-covered one)
    c = cov[~cov.index.isin(exclude)]
    good = c[c >= MIN_COVER]
    pool = good if len(good) else c
    dist_ = pd.Series(abs(pool.index - target), index=pool.index)
    best_ = dist_[dist_ == dist_.min()].index
    return int(max(best_) if prefer_recent else min(best_))
if SHOW_YEARS:
    Y = [int(v) for v in SHOW_YEARS]
else:
    last_year = int(cov.index.max())
    Y = [nearest(2001),
         nearest((2001 + last_year) // 2, exclude=(2012, 2013)),    # 2012-13: Landsat 7 only, striped
         nearest(last_year, prefer_recent=True)]
print("Years shown:", Y, "| share of the city observed:", [f"{cov[y]:.0%}" for y in Y])

# ---------- 2. target grid in Web Mercator (what web maps use) ----------------------------
dash_geom = city_geom.buffer(DASH_BUFFER_M)
to3857 = Transformer.from_crs(CRS, "EPSG:3857", always_xy=True)
to4326 = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
gx, gy = to3857.transform(*dash_geom.exterior.xy)
gb30 = GeoBox.from_bbox((min(gx), min(gy), max(gx), max(gy)), crs="EPSG:3857", resolution=30)
gb60 = gb30.zoom_out(2)
west, south = to4326.transform(gb30.extent.boundingbox.left, gb30.extent.boundingbox.bottom)
east, north = to4326.transform(gb30.extent.boundingbox.right, gb30.extent.boundingbox.top)
BOUNDS = [[south, west], [north, east]]

def rp(da, gb, resampling="bilinear", nodata=None):
    da = assign_crs(da.drop_vars("spatial_ref", errors="ignore"), CRS)
    kw = {"dst_nodata": nodata} if nodata is not None else {}
    return da.odc.reproject(gb, resampling=resampling, **kw).values

dash_city = rasterize([(gpd.GeoSeries([city_geom], crs=CRS).to_crs(3857).iloc[0], 1)],
                      out_shape=gb30.shape, transform=gb30.affine, fill=0).astype(bool)

# ---------- 3. layers for each year -------------------------------------------------------
heat = {y: rp(annual_jj.sel(year=y), gb30) for y in Y}
ndvi_y = {y: rp(cube["ndvi"].sel(time=str(y)).median("time").compute(), gb30) for y in Y}

dash_bbox_4326 = tuple(gpd.GeoSeries([dash_geom], crs=CRS).to_crs(4326).total_bounds)
def bands_year(y, n=3, gb=None, names=("red", "green", "blue", "nir08", "swir16"), res=RES):
    # median of the clearest scenes of that summer, cloud-masked: visible bands for true colour,
    # NIR and SWIR for the land-cover classification. June–July scenes are preferred, so that fields
    # are green in every year (after the harvest they look like bare, built-up land).
    gb = gb30 if gb is None else gb
    iy = inv[inv.index.year == y]
    jj = iy[iy.index.month.isin(JJ)]
    days = set((jj if len(jj) else iy).sort_values("city_clear", ascending=False).index[:n])
    its = [it for it in usable if day(it) in days]
    ds = odc.stac.load(resign(its), bands=list(names) + ["qa_pixel"], bbox=dash_bbox_4326,
                       crs=CRS, resolution=res, groupby="solar_day", chunks={})
    ok = clear(ds["qa_pixel"])
    return {b: rp((ds[b] * 0.0000275 - 0.2).where(ok).median("time").compute(), gb) for b in names}
bands = {}
for y in Y:
    t0 = time.time(); print(f"Loading true-colour bands for {y} ...", end=" ", flush=True)
    bands[y] = bands_year(y); print(f"done in {time.time() - t0:.0f} s", flush=True)
rgb = {y: np.stack([bands[y]["red"], bands[y]["green"], bands[y]["blue"]], axis=-1) for y in Y}

# ---------- 3b. land cover for each year, classified from the same Landsat composites ---------
# Same rules for every year, so differences between years reflect the ground, not the method.
# Simple spectral indices cannot reliably separate sealed surfaces (asphalt, roofs) from bare soil,
# so both form one non-vegetated class; for surface heat, vegetated vs non-vegetated is what matters.
NDVI_TREES, NDVI_VEG, MNDWI_WATER = 0.6, 0.3, 0.10
LU_CLASSES = {1: "Built-up, sealed or bare", 3: "Grass & crops", 4: "Trees & dense vegetation", 5: "Water"}
LU_COLORS = {1: "#c2414b", 3: "#a9d18e", 4: "#2f6b3a", 5: "#2a78d6"}
def classify(b):
    with np.errstate(divide="ignore", invalid="ignore"):
        ndvi = (b["nir08"] - b["red"]) / (b["nir08"] + b["red"])
        mndwi = (b["green"] - b["swir16"]) / (b["green"] + b["swir16"])
    lu = np.zeros(ndvi.shape, "uint8")
    lu[ndvi < NDVI_VEG] = 1
    lu[(ndvi >= NDVI_VEG) & (ndvi < NDVI_TREES)] = 3
    lu[ndvi >= NDVI_TREES] = 4
    lu[(mndwi > MNDWI_WATER) & (ndvi < 0.2)] = 5
    lu[~np.isfinite(ndvi)] = 0
    return lu
lu = {y: classify(bands[y]) for y in Y}

# areas inside the city (Web Mercator pixels are smaller on the ground by cos(latitude))
px_km2 = (30 * np.cos(np.radians((south + north) / 2))) ** 2 / 1e6
lu_stats = {LU_CLASSES[c]: [round(float(((lu[y] == c) & dash_city).sum() * px_km2), 2) for y in Y] for c in LU_CLASSES}
a, z = lu[Y[0]], lu[Y[-1]]
both = dash_city & (a > 0) & (z > 0)
transitions = {
    "veg_to_built": round(float((both & np.isin(a, [3, 4]) & (z == 1)).sum() * px_km2), 2),
    "trees_lost": round(float((both & (a == 4) & np.isin(z, [1, 3])).sum() * px_km2), 2),
    "trees_gained": round(float((both & np.isin(a, [1, 3]) & (z == 4)).sum() * px_km2), 2),
}
print("Land cover in the city (km²):", lu_stats)
print(f"{Y[0]}→{Y[-1]}: vegetation to built-up {transitions['veg_to_built']} km², "
      f"trees lost {transitions['trees_lost']} km², trees gained {transitions['trees_gained']} km²")

# ---------- 4. colour scales (diverging blue-grey-red for heat, single-hue green for NDVI) ---
HEAT_STOPS = ["#0d366b", "#1c5cab", "#3987e5", "#86b6ef", "#f0efec", "#f6b3aa", "#ec7a6e", "#d03b3b", "#8f1d1d"]
NDVI_STOPS = ["#f3f1e7", "#d9e8c5", "#b0d494", "#7cbb63", "#4c9b3f", "#277a2a", "#0e5a1d"]
heat_cmap = mcolors.LinearSegmentedColormap.from_list("heat", HEAT_STOPS)
ndvi_cmap = mcolors.LinearSegmentedColormap.from_list("ndvi", NDVI_STOPS)
HEAT_NORM = mcolors.TwoSlopeNorm(vmin=-4, vcenter=0, vmax=10)
CHANGE_NORM = mcolors.TwoSlopeNorm(vmin=-4, vcenter=0, vmax=4)
NDVI_NORM = mcolors.Normalize(0, 0.8)

def png_uri(rgba, palette=False):
    img = Image.fromarray((np.clip(rgba, 0, 1) * 255).astype("uint8"), "RGBA")
    if palette:                        # colour-mapped layers: 256 colours are plenty, files get much smaller
        img = img.quantize(256, method=Image.Quantize.FASTOCTREE)
    buf = io.BytesIO()
    img.save(buf, "PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def colour(arr, cmap, norm):
    rgba = cmap(norm(np.nan_to_num(arr, nan=0)))
    rgba[..., 3] = np.isfinite(arr)
    return png_uri(rgba, palette=True)

lo, hi = np.nanpercentile(rgb[Y[-1]], [2, 98])
def true_colour(a):
    v = np.clip((a - lo) / (hi - lo), 0, 1) ** (1 / 1.2)
    alpha = np.isfinite(a).all(axis=-1)[..., None].astype(float)
    return png_uri(np.concatenate([np.nan_to_num(v), alpha], axis=-1))

layers = {str(y): {"rgb": true_colour(rgb[y]), "heat": colour(heat[y], heat_cmap, HEAT_NORM),
                   "ndvi": colour(ndvi_y[y], ndvi_cmap, NDVI_NORM)} for y in Y}
for y in Y[1:]:
    layers[str(y)]["change"] = colour(heat[y] - heat[Y[0]], heat_cmap, CHANGE_NORM)
lut = np.zeros((256, 4))
for c, hx in LU_COLORS.items():
    lut[c] = list(mcolors.to_rgb(hx)) + [1]
for y in Y:
    layers[str(y)]["lu"] = png_uri(lut[lu[y]], palette=True)

# ---------- 5. 60 m value grids for the click inspector -----------------------------------
def enc_f(a):
    return base64.b64encode(np.where(np.isfinite(a), np.round(a * 100), -32768).astype("<i2").tobytes()).decode()
def enc_u8(a):
    return base64.b64encode(np.nan_to_num(a, nan=0).astype("uint8").tobytes()).decode()

grids = {f"heat_{y}": enc_f(rp(annual_jj.sel(year=y), gb60)) for y in Y}
grids.update({f"ndvi_{y}": enc_f(rp(cube["ndvi"].sel(time=str(y)).median("time").compute(), gb60)) for y in Y})
grids["dens"] = enc_f(rp(built, gb60, "average"))
grids["lc"] = enc_u8(rp(landcover.astype("uint8"), gb60, "nearest", nodata=0))
for y in Y:   # 60 m land cover per year: majority of the 2 x 2 block of 30 m pixels
    h2, w2 = gb60.shape
    pad = np.zeros((h2 * 2, w2 * 2), "uint8")
    hh, ww = min(h2 * 2, lu[y].shape[0]), min(w2 * 2, lu[y].shape[1])
    pad[:hh, :ww] = lu[y][:hh, :ww]
    blk = pad.reshape(h2, 2, w2, 2).transpose(0, 2, 1, 3).reshape(h2, w2, 4)
    counts = np.stack([(blk == c).sum(-1) for c in range(6)], -1); counts[..., 0] = 0
    grids[f"lu_{y}"] = enc_u8(np.where(counts.max(-1) > 0, counts.argmax(-1), 0))
grids["lcz"] = enc_u8(rp(lcz.fillna(0).astype("uint8"), gb60, "nearest", nodata=0)) if lcz is not None else None
dist_names = []
if districts is not None:
    d3857 = districts.to_crs(3857).reset_index(drop=True)
    dist_names = list(d3857["name"].astype(str))
    grids["district"] = enc_u8(rasterize([(g, i + 1) for i, g in enumerate(d3857.geometry)],
                                         out_shape=gb60.shape, transform=gb60.affine, fill=0))
GRID = {"x0": gb60.affine.c, "y0": gb60.affine.f, "res": gb60.resolution.x, "w": gb60.shape[1], "h": gb60.shape[0]}

# ---------- 6. places to inspect: 300 m cells with the largest warming and cooling ---------
k = 10
def coarse(a):
    h, w = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.nanmean(a[:h, :w].reshape(h // k, k, w // k, k), axis=(1, 3))
def coarse_mode(a):
    h, w = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    blk = a[:h, :w].reshape(h // k, k, w // k, k)
    counts = np.stack([(blk == c).sum(axis=(1, 3)) for c in range(6)], -1); counts[..., 0] = 0
    return counts.argmax(-1)
lu_mode = {y: coarse_mode(lu[y]) for y in Y}
dh = coarse(heat[Y[-1]] - heat[Y[0]])
dn = coarse(ndvi_y[Y[-1]] - ndvi_y[Y[0]])
incity = coarse(dash_city.astype(float)) > 0.6
def pick(score, n, min_cells=3):
    order = np.argsort(np.where(incity & np.isfinite(score), score, -np.inf), axis=None)[::-1]
    chosen = []
    for idx in order:
        r, c = np.unravel_index(idx, score.shape)
        if not np.isfinite(score[r, c]) or not incity[r, c]: break
        if all(abs(r - r2) >= min_cells or abs(c - c2) >= min_cells for r2, c2 in chosen):
            chosen.append((r, c))
        if len(chosen) == n: break
    out = []
    for r, c in chosen:
        x = gb30.affine.c + (c + 0.5) * k * 30
        y = gb30.affine.f - (r + 0.5) * k * 30
        lon, lat = to4326.transform(x, y)
        out.append({"lat": lat, "lon": lon, "dheat": round(float(dh[r, c]), 2), "dndvi": round(float(dn[r, c]), 3),
                    "lu0": int(lu_mode[Y[0]][r, c]), "lu1": int(lu_mode[Y[-1]][r, c])})
    return out
hotspots = {"warming": pick(dh, N_HOTSPOTS), "cooling": pick(-dh, max(3, N_HOTSPOTS // 2))}

# ---------- 7. charts, districts, key numbers ------------------------------------------------
xs = np.array([yearly.index.min(), yearly.index.max()])
charts = {
    "years": [int(v) for v in yearly.index], "suhi": [round(float(v), 2) for v in yearly.suhi],
    "suhiMonth": {name: [None if pd.isna(v) else round(float(v), 2)
                         for v in series["suhi"][series.index.month == m].groupby(series.index[series.index.month == m].year)
                         .median().reindex(yearly.index)]
                  for m, name in ((6, "June"), (7, "July"), (8, "August"))},
    "periods": list(class_table.columns),
    "classes": {n: [None if pd.isna(v) else round(float(v), 2) for v in class_table.loc[n]] for n in CLASSES.values()},
    "districts": [] if districts is None else
        [{"name": r.district, "value": float(r[f"anomaly_{p1}"])} for _, r in dist.head(15).iterrows()],
}
dgeo = None
if districts is not None:
    dg = districts.to_crs(3857).copy()
    dg["geometry"] = dg.geometry.simplify(15)
    dg = dg.merge(dist.rename(columns={"district": "name"}), on="name", how="left").to_crs(4326)
    dgeo = json.loads(dg[["name", f"anomaly_{p1}", "change", "green_share", "geometry"]]
                      .rename(columns={f"anomaly_{p1}": "anomaly"}).to_json())
kpis = [
    {"label": "Heat island, June / July / August",
     "value": " / ".join(f"{v:.1f}" for v in series["suhi"].groupby(series.index.month).median().reindex([6, 7, 8])) + " °C",
     "note": "median of all clear scenes"},
]
if "month_adj" in globals():          # month-adjusted trends from section 5b
    kpis += [
        {"label": "Heat island trend, same month", "value": f"{month_adj.iloc[2]['trend_per_decade']:+.2f} °C / decade",
         "note": f"p = {month_adj.iloc[2]['p']:.2f}"},
        {"label": "City surface warming", "value": f"{month_adj.iloc[0]['trend_per_decade']:+.2f} °C / decade",
         "note": f"absolute LST, month-adjusted, p = {month_adj.iloc[0]['p']:.2f}"}]
else:
    kpis += [{"label": "Heat island trend", "value": f"{slope*10:+.2f} °C / decade", "note": f"p = {p:.2f}"}]
kpis += [
    {"label": "Warmest land cover", "value": hot, "note": f"{class_table.loc[hot, p1]:+.1f} °C vs area mean"},
    {"label": "Cooling by vegetation", "value": f"{float(reg_table.loc[p1, 'slope_per_0.1_NDVI_C']):+.2f} °C", "note": "per +0.1 NDVI"},
    {"label": f"Built-up and sealed land, {Y[0]}–{Y[-1]}",
     "value": f"{lu_stats[LU_CLASSES[1]][0]:.1f} → {lu_stats[LU_CLASSES[1]][-1]:.1f} km²",
     "note": f"inside the city; {transitions['veg_to_built']:.1f} km² of vegetation converted (gross)"},
]
# ---------- 7b. year-by-year viewer: one heat map (and true-colour image) per summer --------------
# Written as separate files in years/ next to the dashboard, loaded only when shown.
YEAR_DIRS = [f"{OUT}/years", "docs/years"]
for d_ in YEAR_DIRS:
    os.makedirs(d_, exist_ok=True)
def save_png(rgba, name):
    img = Image.fromarray((np.clip(rgba, 0, 1) * 255).astype("uint8"), "RGBA").quantize(256, method=Image.Quantize.FASTOCTREE)
    for d_ in YEAR_DIRS:
        img.save(f"{d_}/{name}", optimize=True)
frames = []
for y in YEARS_ALL:
    h = rp(annual_jj.sel(year=y), gb60)
    rgba = heat_cmap(HEAT_NORM(np.nan_to_num(h, nan=0))); rgba[..., 3] = np.isfinite(h)
    save_png(rgba, f"heat_{y}.png")
    fr = Image.fromarray((rgba * 255).astype("uint8"), "RGBA")
    bg = Image.new("RGBA", fr.size, (255, 255, 255, 255)); bg.alpha_composite(fr)
    from PIL import ImageDraw
    ImageDraw.Draw(bg).text((12, 10), f"{y}  ({BASIS[y].split(' (')[0]}, {NIMG[y]} images)", fill=(20, 20, 20, 255))
    frames.append(bg.convert("P", palette=Image.ADAPTIVE))
for d_ in YEAR_DIRS:
    frames[0].save(f"{d_}/heat_by_year.gif", save_all=True, append_images=frames[1:], duration=700, loop=0)
annual_rgb = []
if ANNUAL_RGB:
    lo_, hi_ = lo, hi
    for y in YEARS_ALL:
        t0 = time.time()
        print(f"  true colour {y}: loading ...", end=" ", flush=True)
        try:
            b = bands_year(y, n=2, gb=gb60, names=("red", "green", "blue"), res=RES * 2)
        except Exception as e:
            print(f"skipped ({type(e).__name__})", flush=True)
            continue
        a_ = np.stack([b["red"], b["green"], b["blue"]], axis=-1)
        v = np.clip((a_ - lo_) / (hi_ - lo_), 0, 1) ** (1 / 1.2)
        img = Image.fromarray((np.nan_to_num(v, nan=1.0) * 255).astype("uint8"), "RGB")
        for d_ in YEAR_DIRS:
            img.save(f"{d_}/rgb_{y}.jpg", quality=82, optimize=True)
        annual_rgb.append(y)
        print(f"done in {time.time() - t0:.0f} s", flush=True)
annual_info = {"years": YEARS_ALL, "basis": {str(y): BASIS[y] for y in YEARS_ALL},
               "n": {str(y): NIMG[y] for y in YEARS_ALL}, "rgb": annual_rgb, "dir": "years/"}
print(f"Year-by-year viewer: {len(YEARS_ALL)} heat maps, {len(annual_rgb)} true-colour images, GIF in {OUT}/years/")

payload = {"years": Y, "bounds": BOUNDS, "layers": layers, "grids": grids, "grid": GRID,
           "classes": {str(c): n for c, n in CLASSES.items()}, "lcz": {str(c): n for c, n in LCZ_NAMES.items()},
           "districtNames": dist_names, "districts": dgeo, "hotspots": hotspots, "charts": charts, "kpis": kpis,
           "heatStops": HEAT_STOPS, "ndviStops": NDVI_STOPS,
           "luClasses": {str(c): n for c, n in LU_CLASSES.items()}, "luColors": {str(c): v for c, v in LU_COLORS.items()},
           "luStats": lu_stats, "transitions": transitions, "annual": annual_info,
           "center": [(south + north) / 2, (west + east) / 2], "maptilerKey": MAPTILER_KEY}

# ---------- 8. write the HTML ------------------------------------------------------------------
TEMPLATE = open("dashboard_template.html", encoding="utf-8").read() if os.path.exists("dashboard_template.html") else DASHBOARD_TEMPLATE
html = TEMPLATE.replace("__PAYLOAD__", json.dumps(payload, separators=(",", ":")))
path = f"{OUT}/brno_heat_dashboard.html"
with open(path, "w", encoding="utf-8") as f:
    f.write(html)
os.makedirs("docs", exist_ok=True)                      # GitHub Pages serves docs/index.html
with open("docs/index.html", "w", encoding="utf-8") as f:
    f.write(html)
print(f"Dashboard written to {path} ({os.path.getsize(path) / 1e6:.1f} MB). Open it in a web browser.")


## 12. Comparison with previous studies
Three levels, from strongest to weakest:

1. **Same scenes as Dobrovolný (2013).** His two Landsat scenes (24 May 2001, Landsat 7; 15 June 2006, Landsat 5) are processed with this notebook's pipeline and his definitions: class-based SUHI, water deviation, NDVI and DENS regressions on the 300 m grid. Matching his 4.2 and 6.7 °C validates the processing; a difference points to method choices (OSM classes instead of his classification, Collection 2 temperature algorithm, resolution), not to the time series.
2. **Equivalent metrics** for every published value: class deviations, regressions, LCZ ranking (Geletič et al. 2016), cooling per NDVI (Kirschner et al. 2024, Prague).
3. **Context only**: air-temperature heat island from stations (Janků & Dobrovolný 2022) and absolute LST trends (Kuzevičová et al. 2026, Košice) measure different things and are not directly comparable.

The table is saved as `comparison_with_studies.csv`, ready for the report.

In [ ]:
# ---------- 12a. Same scenes as Dobrovolný (2013): 24 May 2001 (Landsat 7) and 15 June 2006 (Landsat 5) ----------
REF = {"2001-05-24": {"suhi": 4.2, "water": -3.2}, "2006-06-15": {"suhi": 6.7, "water": -8.0}}

def scene_metrics(date):
    its = [it for it in catalog.search(collections=["landsat-c2-l2"], bbox=STUDY_BBOX, datetime=date).items()
           if it.properties.get("landsat:collection_category") == "T1"]
    if not its:
        print(f"{date}: scene not found in the catalogue"); return None
    plat = its[0].properties["platform"]
    ds = load_cube(its, "lwir11" if plat in ("landsat-8", "landsat-9") else "lwir")
    ds = ds.isel(time=0).reindex_like(cube.isel(time=0), method="nearest", tolerance=RES / 2)
    t = ds["lst"].where(in_study)
    area_mean = float(t.where(landcover > 0).mean())
    out = {"date": date, "platform": plat,
           "observed_share": round(float(t.notnull().sum() / in_study.sum()), 2),
           "suhi_classes": float(t.where(landcover.isin([1, 2])).mean() - t.where(landcover.isin([4, 5])).mean()),
           "industry_vs_mean": float(t.where(landcover == 2).mean()) - area_mean,
           "water_vs_mean": float(t.where(landcover == 6).mean()) - area_mean}
    g = xr.Dataset({"lst": to_grid(ds["lst"]), "ndvi": to_grid(ds["ndvi"]), "dens": dens})
    g = g.where(cell_city & ~cell_water).to_dataframe()[["lst", "ndvi", "dens"]].dropna()
    if len(g) > 30:
        out["r2_ndvi"] = stats.linregress(g.ndvi, g.lst).rvalue ** 2
        out["r2_dens"] = stats.linregress(g.dens, g.lst).rvalue ** 2
    return out

same_date = [m for m in (scene_metrics(d) for d in REF) if m]
same_date_df = pd.DataFrame(same_date).round(2)
display(same_date_df)

# ---------- 12b. Comparison table with all studies ----------
def f(v, nd=1, unit=" °C", sign=True):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "–"
    return (f"{v:+.{nd}f}" if sign else f"{v:.{nd}f}") + unit

rows = []
for m in same_date:
    r = REF[m["date"]]
    rows += [
        ["Dobrovolný (2013)", f"SUHI from land-cover classes, {m['date']}", f"{r['suhi']:.1f} °C",
         f(m["suhi_classes"], sign=False), "Yes: same scene and same definition; classes from OSM instead of his own classification"],
        ["Dobrovolný (2013)", f"Water vs area mean, {m['date']}", f"{r['water']:+.1f} °C",
         f(m["water_vs_mean"]), "Yes: same scene"],
        ["Dobrovolný (2013)", f"R² NDVI / R² DENS, 300 m grid, {m['date']}", "0.56–0.67 / 0.37–0.40",
         f"{m.get('r2_ndvi', np.nan):.2f} / {m.get('r2_dens', np.nan):.2f}", "Yes: same grid; DENS from today's OSM buildings"],
    ]
ind_dev = class_table.loc["Industry & transport", p1]
rows += [
    ["Dobrovolný (2013)", "SUHI from classes, multi-summer median", "4.2 / 6.7 °C (single dates)",
     f"{suhi_cls[p0]:.1f} → {suhi_cls[p1]:.1f} °C ({p0} → {p1})", "Order of magnitude only: medians of many summers are lower than single clear days"],
    ["Dobrovolný (2013)", "Industry vs area mean", "+3 to +5 °C", f(ind_dev) + f" ({p1})", "Yes: same definition, different period"],
    ["Dobrovolný (2013)", "R² NDVI / R² DENS, 300 m grid", "0.56–0.67 / 0.37–0.40",
     f"{reg_table.loc[p1, 'R2_NDVI']:.2f} / {reg_table.loc[p1, 'R2_DENS']:.2f} ({p1})", "Yes: same method, multi-summer medians"],
    ["Geletič et al. (2016)", "Warmest / coolest Local Climate Zone in Brno", "LCZ 10 heavy industry / G water, A dense trees",
     "–" if lcz is None else f"{lcz_table.index[-1]} / {lcz_table.index[0]}", "Ranking only (qualitative)"],
    ["Kirschner et al. (2024), Prague", "Cooling per +0.1 NDVI", "−1.0 to −1.5 °C",
     f(float(reg_table.loc[p1, "slope_per_0.1_NDVI_C"]), 2), "Partly: another city, same physics"],
    ["Kirschner et al. (2024), Prague", "R² of NDVI", "0.38–0.54", f"{reg_table.loc[p1, 'R2_NDVI']:.2f}", "Partly: another city and grid"],
    ["Janků & Dobrovolný (2022)", "Daytime heat island", "1.93 °C (air temperature, stations)",
     f(period_mean("suhi", *last[1]), sign=False) + " (surface)", "No: air vs surface; surface heat islands are larger by nature"],
    ["Kuzevičová et al. (2026), Košice", "Warming of absolute urban LST", "+5.83 °C over 1985–2025",
     f(u_slope * 10, 2, " °C/decade"), "Qualitative only: absolute LST mixes city effect and weather of each scene"],
]
comparison = pd.DataFrame(rows, columns=["Study", "Metric", "Published", "This study", "Comparable?"])
comparison.to_csv(f"{OUT}/comparison_with_studies.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(comparison)


## 13. Figures for the report
The three figures of the 2-page report, numbered as in the report: `REPORT_Figure_1.png` (heat island of every clear image by month), `REPORT_Figure_2.png` (heat map and Local Climate Zones) and `REPORT_Figure_3.png` (example of local change, the place that warmed most). The other figures (`fig0`–`fig9`) belong to the notebook only.

In [ ]:
# Writes the three figures of the 2-page report with the same numbers as the report text:
#   outputs/REPORT_Figure_1.png  heat island of every clear image by month + month mix per period
#   outputs/REPORT_Figure_2.png  (a) heat map 2020–2026, (b) heat by Local Climate Zone
#   outputs/REPORT_Figure_3.png  example of local change (true colour and heat, first vs last year)
import matplotlib.colors as mcolors
REPORT_FIGS = {}
MONTHS = {6: ("June", "#2a78d6"), 7: ("July", "#eb6834"), 8: ("August", "#1baf7a")}
HEAT_STOPS_R = ["#0d366b", "#1c5cab", "#3987e5", "#86b6ef", "#f0efec", "#f6b3aa", "#ec7a6e", "#d03b3b", "#8f1d1d"]
cm_heat = mcolors.LinearSegmentedColormap.from_list("heat", HEAT_STOPS_R)
nm_heat = mcolors.TwoSlopeNorm(vmin=-4, vcenter=0, vmax=10)
rc = {"font.size": 7.5, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
      "grid.color": "#e6e5e0", "grid.linewidth": 0.5, "axes.axisbelow": True}

# ---- Figure 1 -------------------------------------------------------------------------------
with plt.rc_context(rc):
    d1 = series.copy(); d1["m"] = d1.index.month
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.1, 2.15), gridspec_kw={"width_ratios": [2.25, 1]})
    for m, (lab, c) in MONTHS.items():
        s_ = d1[d1.m == m]
        a1.scatter(s_.year, s_.suhi, s=10, color=c, edgecolor="white", linewidth=0.5, zorder=3, label=lab)
    jj = d1[d1.m.isin([6, 7])].groupby("year").suhi.median()
    a1.plot(jj.index, jj.values, color="#2b2b2a", lw=1.1, zorder=4, label="June–July, yearly median")
    sl_, ic_, _, _ = stats.theilslopes(jj.values, jj.index)
    a1.plot([jj.index.min(), jj.index.max()], [ic_ + sl_ * jj.index.min(), ic_ + sl_ * jj.index.max()],
            color="#2b2b2a", lw=1.1, ls=(0, (4, 3)), zorder=4, label="June–July trend")
    a1.axhline(0, color="#8a8986", lw=0.6)
    a1.set(ylabel="City minus countryside (°C)")
    a1.set_title("a  Heat island of every clear image, by month", loc="left", fontsize=8, fontweight="bold")
    cnt = pd.crosstab(pd.cut(d1.year, [y[0] - 1 for y in PERIODS.values()] + [list(PERIODS.values())[-1][1]],
                             labels=list(PERIODS)), d1.m).reindex(columns=[6, 7, 8], fill_value=0)
    share = cnt.div(cnt.sum(axis=1), axis=0) * 100
    left = np.zeros(len(share))
    for m, (lab, c) in MONTHS.items():
        a2.barh(share.index.astype(str), share[m], left=left, color=c, edgecolor="white", height=0.62)
        for i, v in enumerate(share[m]):
            if v >= 14:
                a2.text(left[i] + v / 2, i, f"{v:.0f}", ha="center", va="center", fontsize=6.5, color="white", fontweight="bold")
        left += share[m].values
    for i, n_ in enumerate(cnt.sum(axis=1)):
        a2.text(101, i, f"n={n_}", va="center", fontsize=6.5, color="#52514e")
    a2.invert_yaxis(); a2.set_xlim(0, 100); a2.grid(False); a2.set_xlabel("Share of images (%)")
    a2.set_title("b  Month of the images per period", loc="left", fontsize=8, fontweight="bold")
    h_, l_ = a1.get_legend_handles_labels()
    fig.legend(h_, l_, frameon=False, ncol=5, fontsize=7, loc="lower center", bbox_to_anchor=(0.5, 0.0), columnspacing=1.8)
    plt.tight_layout(w_pad=1.2, rect=(0, 0.08, 1, 1))
    REPORT_FIGS[1] = f"{OUT}/REPORT_Figure_1.png"; plt.savefig(REPORT_FIGS[1], dpi=300); plt.show()

# ---- Figure 2 -------------------------------------------------------------------------------
with plt.rc_context(rc):
    fig = plt.figure(figsize=(6.4, 2.75))
    a1 = fig.add_axes([0.0, 0.03, 0.44, 0.8]); cax = fig.add_axes([0.445, 0.12, 0.012, 0.7])
    a2 = fig.add_axes([0.66, 0.16, 0.33, 0.68])
    im = a1.imshow(anom_period[p1].values, cmap=cm_heat, norm=nm_heat, interpolation="nearest",
                   extent=[float(anom_period[p1].x.min()), float(anom_period[p1].x.max()),
                           float(anom_period[p1].y.min()), float(anom_period[p1].y.max())])
    city.boundary.plot(ax=a1, color="k", lw=0.7)
    a1.set_axis_off(); a1.set_title(f"a  Heat above the countryside, {p1}", loc="left", fontsize=8, fontweight="bold")
    cb = fig.colorbar(im, cax=cax, ticks=[-4, 0, 4, 8, 10]); cb.ax.tick_params(labelsize=6.5); cb.set_label("°C", fontsize=7)
    if lcz is not None:
        t_ = lcz_table[lcz_table.pixels >= 200].sort_values(p1)
        a2.barh(t_.index, t_[p1], color=["#e34948" if v > 0 else "#2a78d6" for v in t_[p1]], height=0.66, edgecolor="white")
        for i, v in enumerate(t_[p1]):
            a2.text(v + (0.25 if v > 0 else -0.25), i, f"{v:+.1f}", va="center", ha="left" if v > 0 else "right", fontsize=6.3)
        a2.axvline(0, color="#8a8986", lw=0.6); a2.set_xlim(-8, 8); a2.grid(axis="y", visible=False)
        a2.tick_params(axis="y", length=0, labelsize=6.5); a2.set_xlabel(f"°C above countryside, {p1}", fontsize=7)
        a2.set_title("b  Heat by Local Climate Zone", loc="left", fontsize=8, fontweight="bold", x=-0.6)
    REPORT_FIGS[2] = f"{OUT}/REPORT_Figure_2.png"; plt.savefig(REPORT_FIGS[2], dpi=300); plt.show()

# ---- Figure 3: the place that warmed most (first entry of the dashboard list) -----------------
try:
    spot = hotspots["warming"][0]
    to3857_ll = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
    sx, sy = to3857_ll.transform(spot["lon"], spot["lat"])
    col, row = ~gb30.affine * (sx, sy); col, row = int(col), int(row)
    half = 40                                                          # 40 px x 30 m = 1.2 km each side
    win = (slice(max(row - half, 0), row + half), slice(max(col - half, 0), col + half))
    def tc(a):
        v = np.clip((a[win] - lo) / (hi - lo), 0, 1) ** (1 / 1.2); return np.nan_to_num(v, nan=1)
    panels = [(tc(rgb[Y[0]]), f"True colour {Y[0]}"), (tc(rgb[Y[-1]]), f"True colour {Y[-1]}"),
              (heat[Y[0]][win], f"Heat {Y[0]}"), (heat[Y[-1]][win], f"Heat {Y[-1]}")]
    with plt.rc_context({"font.size": 7.5}):
        fig, axs = plt.subplots(1, 4, figsize=(5.4, 1.75))
        for ax, (arr, lab) in zip(axs, panels):
            if arr.ndim == 3: ax.imshow(arr)
            else: imh = ax.imshow(arr, cmap=cm_heat, norm=nm_heat)
            ax.set_xticks([]); ax.set_yticks([]); ax.set_title(lab, fontsize=7, loc="left", pad=2)
            for sp in ax.spines.values(): sp.set_visible(True); sp.set_color("#c3c2b7"); sp.set_linewidth(0.5)
        plt.tight_layout(w_pad=0.3, rect=(0, 0.2, 1, 0.97))
        cax = fig.add_axes([axs[2].get_position().x0, 0.13, axs[3].get_position().x1 - axs[2].get_position().x0, 0.04])
        cb = fig.colorbar(imh, cax=cax, orientation="horizontal", ticks=[-4, 0, 4, 8, 10])
        cb.ax.tick_params(labelsize=6, length=2, pad=1); cb.set_label("°C above countryside", fontsize=6, labelpad=1)
        REPORT_FIGS[3] = f"{OUT}/REPORT_Figure_3.png"; plt.savefig(REPORT_FIGS[3], dpi=300); plt.show()
    spot_district = None
    if districts is not None:
        pt = gpd.GeoSeries(gpd.points_from_xy([spot["lon"]], [spot["lat"]]), crs=4326).to_crs(CRS).iloc[0]
        hit = districts[districts.contains(pt)]
        spot_district = None if hit.empty else str(hit["name"].iloc[0])
    print(f"Figure 3 location: {spot['lat']:.4f} N, {spot['lon']:.4f} E, district {spot_district}; "
          f"heat change {spot['dheat']:+.1f} °C; land cover {LU_CLASSES.get(spot['lu0'])} -> {LU_CLASSES.get(spot['lu1'])}")
except Exception as e:
    print("Figure 3 skipped (needs section 11):", e)
print("Report figures:", REPORT_FIGS)


## 13b. What each reported number means
Every number used in the report, with the images, period, aggregation and statistical model behind it (`outputs/results_definitions.csv`). Heat island = median LST of built-up city pixels minus median LST of farmland/grassland in the 2–12 km ring at the city's elevation, computed **per image**. *Yearly median* = median of the per-image values of one summer. *Period value* = mean of the yearly medians of the summers in that period. *Theil–Sen* = median of all pairwise slopes between yearly medians, with the Mann–Kendall test for its p-value. *Month-adjusted OLS* = ordinary least squares on all per-image values with one intercept per month and a common year slope, so each month is compared only with itself.

In [ ]:
JJ_ROW = "Cropland & grass, same elevation (main), June–July"
def rget(row, col):
    return float(robust.loc[row, col]) if row in robust.index else np.nan
d_rows = [
    ("Heat island 2000–2004 / 2020–2026 (main result)", f"{rget(JJ_ROW, 'first_period'):.1f} / {rget(JJ_ROW, 'last_period'):.1f} °C",
     "June–July images", "2000–2004 and 2020–2026", "mean of yearly medians of per-image heat island", "descriptive"),
    ("Heat island trend (main result)", f"{rget(JJ_ROW, 'trend_per_decade'):+.2f} °C/decade, p = {rget(JJ_ROW, 'p'):.2f}",
     "June–July images", f"2000–2026 ({int(rget(JJ_ROW, 'years'))} summers)", "yearly medians", "Theil–Sen slope, Mann–Kendall p"),
    ("Heat island trend, month-adjusted check", f"{month_adj.iloc[2]['trend_per_decade']:+.2f} ± {month_adj.iloc[2]['se_per_decade']:.2f} °C/decade, p = {month_adj.iloc[2]['p']:.2f}",
     "all June–August images", f"2000–2026 ({int(month_adj.iloc[2]['scenes'])} images)", "per-image values", "OLS with month intercepts, t-test p"),
    ("Heat island 2000–2004 / 2020–2026, all months (shown as misleading)", f"{period_mean('suhi', *first[1]):.1f} / {period_mean('suhi', *last[1]):.1f} °C",
     "all June–August images", "2000–2004 and 2020–2026", "mean of yearly medians", "descriptive"),
    ("Urban surface temperature trend (absolute)", f"{urban_jj['trend_per_decade']:+.2f} °C/decade, p = {urban_jj['p']:.2f}",
     "June–July images", "2000–2026", "yearly medians of urban LST", "Theil–Sen slope, Mann–Kendall p"),
    ("Urban surface temperature trend (absolute), month-adjusted", f"{month_adj.iloc[0]['trend_per_decade']:+.2f} ± {month_adj.iloc[0]['se_per_decade']:.2f} °C/decade, p = {month_adj.iloc[0]['p']:.2f}",
     "all June–August images", "2000–2026", "per-image urban LST", "OLS with month intercepts, t-test p"),
    ("Farmland surface temperature trend (absolute), month-adjusted", f"{month_adj.iloc[1]['trend_per_decade']:+.2f} ± {month_adj.iloc[1]['se_per_decade']:.2f} °C/decade, p = {month_adj.iloc[1]['p']:.2f}",
     "all June–August images", "2000–2026", "per-image rural LST", "OLS with month intercepts, t-test p"),
    ("City minus forest trend, June–July", f"{rget('Forest only, June–July', 'trend_per_decade'):+.2f} °C/decade, p = {rget('Forest only, June–July', 'p'):.2f}",
     "June–July images", "2000–2026", "yearly medians", "Theil–Sen slope, Mann–Kendall p"),
    ("City minus forest trend, month-adjusted", f"{month_adj.iloc[4]['trend_per_decade']:+.2f} ± {month_adj.iloc[4]['se_per_decade']:.2f} °C/decade, p = {month_adj.iloc[4]['p']:.2f}",
     "all June–August images", "2000–2026", "per-image values", "OLS with month intercepts, t-test p"),
    ("Heat by Local Climate Zone", "see lcz_anomaly.csv", "all June–August images", p1,
     "pixel median of the period, then mean per zone", "descriptive"),
    ("NDVI association (R² and slope)", f"R² = {float(reg_table.loc[p1, 'R2_NDVI']):.2f}, {float(reg_table.loc[p1, 'slope_per_0.1_NDVI_C']):+.2f} °C per +0.1 NDVI",
     "all June–August images", f"{p1}, {int(reg_table.loc[p1, 'cells'])} cells of 300 m", "period median per pixel, mean per cell",
     "simple OLS, NDVI only (association, not causal)"),
    ("NDVI with building density held constant", f"R² = {float(multi_table.loc[p1, 'R2_NDVI_plus_DENS']):.2f}, {float(multi_table.loc[p1, 'NDVI_per_0.1_C_holding_DENS']):+.2f} °C per +0.1 NDVI",
     "all June–August images", p1, "as above", "multiple OLS, NDVI + DENS"),
]
defs_table = pd.DataFrame(d_rows, columns=["result", "value", "images", "period", "aggregation", "statistical model"])
defs_table.to_csv(f"{OUT}/results_definitions.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(defs_table)

## 14. Summary and download
All key numbers in one table (`summary.csv`), and every figure and table zipped for download.

In [ ]:
summary = {
    "usable_scenes": int(len(series)),
    "suhi_first_period_C": round(period_mean("suhi", *first[1]), 2),
    "suhi_last_period_C": round(period_mean("suhi", *last[1]), 2),
    "suhi_trend_C_per_decade": round(slope * 10, 2),
    "suhi_trend_p_value": round(p, 3),
    "urban_lst_trend_C_per_decade": round(u_slope * 10, 2),
    "robust_trend_min_max_C_per_decade": f"{robust['trend_per_decade'].min():.2f} to {robust['trend_per_decade'].max():.2f}",
    "robust_variants_positive": f"{int((robust['trend_per_decade'] > 0).sum())} of {len(robust)}",
    "june_july_suhi_first_period_C": round(float(robust.loc["Cropland & grass, same elevation (main), June–July", "first_period"]), 2),
    "june_july_suhi_last_period_C": round(float(robust.loc["Cropland & grass, same elevation (main), June–July", "last_period"]), 2),
    "june_july_suhi_trend_C_per_decade": round(float(robust.loc["Cropland & grass, same elevation (main), June–July", "trend_per_decade"]), 2),
    "june_july_suhi_trend_p": round(float(robust.loc["Cropland & grass, same elevation (main), June–July", "p"]), 2),
    "june_july_urban_lst_trend_C_per_decade": round(float(urban_jj["trend_per_decade"]), 2),
    "month_adjusted_suhi_trend_C_per_decade": round(float(month_adj.iloc[2]["trend_per_decade"]), 2),
    "month_adjusted_suhi_trend_p": round(float(month_adj.iloc[2]["p"]), 3),
    "month_adjusted_urban_lst_trend_C_per_decade": round(float(month_adj.iloc[0]["trend_per_decade"]), 2),
    "month_adjusted_forest_suhi_trend_C_per_decade": round(float(month_adj.iloc[4]["trend_per_decade"]), 2),
    "month_adjusted_forest_suhi_trend_p": round(float(month_adj.iloc[4]["p"]), 3),
    "class_suhi_first_period_C": round(suhi_cls[p0], 2),
    "class_suhi_last_period_C": round(suhi_cls[p1], 2),
    "warmest_class_last_period": hot,
    "warmest_class_deviation_C": round(class_table.loc[hot, p1], 2),
    "warmest_lcz": None if lcz is None else lcz_table.index[-1],
    "grid_r2_ndvi_last": float(reg_table.loc[p1, "R2_NDVI"]),
    "grid_r2_dens_last": float(reg_table.loc[p1, "R2_DENS"]),
    "cooling_per_0.1_ndvi_C": float(reg_table.loc[p1, "slope_per_0.1_NDVI_C"]),
    "rural_reference_elevation_m": round(z0),
    "landcover_source": LC_SOURCE,
    "building_density_source": DENS_SOURCE,
}
pd.Series(summary).to_csv(f"{OUT}/summary.csv", header=["value"])
print(pd.Series(summary).to_string())

with zipfile.ZipFile("brno_suhi_outputs.zip", "w") as z:
    for root, _, fs in os.walk(OUT):
        for f in fs:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, OUT))
try:
    from google.colab import files
    files.download("brno_suhi_outputs.zip")
except ImportError:
    print("Outputs saved in", OUT)

### Notes for interpretation
* Landsat measures **surface** temperature at ≈10:00 local time, not air temperature, and not the afternoon or night peak of the heat island.
* Landsat 5, 7 and 8/9 have slightly different thermal sensors; using an urban–rural **difference** reduces the effect of these offsets.
* 2012–2013 rely on Landsat 7 only (data gaps), and Landsat 7 is dropped after 2016 because of its orbit drift.
* WorldCover (2021) defines urban and rural masks for the whole period: areas urbanised after 2000 count as urban throughout, which, if anything, understates change in new developments.
* OpenStreetMap land use and buildings describe the **current** city. They are used for every period, so changes in land use over time are not captured by the classes or by building density.
* Land-cover classes are rebuilt from OpenStreetMap, not from Dobrovolný's own classification, so his values are a reference for order of magnitude rather than an exact benchmark.